# Stage 1 - loading, tokenization, blocking, base features
Executed development notebook (outputs stripped, install cells removed). Run top to bottom before `src/pipeline_v5.py`.
Produces: token tables, `raw_strings_*`, `training_matrix`, `val_matrix_with_tfidf`, `candidates_final_test`, `test_matrix_full`, `candidate_pairs.tsv`.
Cells that re-check or re-derive intermediate results were kept as they were executed.

In [ ]:
import pandas as pd
import os
import numpy as np
import re
from difflib import SequenceMatcher

# SageMaker notebook instances have no /kaggle/input mount -- point BASE
# at the local copy downloaded in Cell 0. Verify the exact subfolder name
# with the `find` output above and adjust if it differs.
DATA_ROOT = "/home/ec2-user/SageMaker/data"
os.makedirs(DATA_ROOT, exist_ok=True)
BASE = f"{DATA_ROOT}/dataset"


s1 = pd.read_csv(f"{BASE}/train/train_source1.tsv", sep="\t")
s2 = pd.read_csv(f"{BASE}/train/train_source2.tsv", sep="\t")
s3 = pd.read_csv(f"{BASE}/train/train_source3.tsv", sep="\t")
gt = pd.read_csv(f"{BASE}/train/train_ground_truth.tsv", sep="\t")

gt['matched_entity_ids'] = gt['matched_entity_ids'].fillna('')
gt['match_list'] = gt['matched_entity_ids'].apply(
    lambda x: [i.strip() for i in x.split(',') if i.strip() != '']
)
gt['n_matches'] = gt['match_list'].apply(len)

s1_idx = s1.set_index('entity_id')
s2_idx = s2.set_index('entity_id')
s3_idx = s3.set_index('entity_id')

In [ ]:
import unicodedata
import regex  # pip install regex --break-system-packages, if not already installed

TOKEN_RE = regex.compile(r'[\p{L}\p{M}\p{N}]+')

def tokenize(s):
    if pd.isnull(s):
        return []
    s = unicodedata.normalize('NFC', str(s)).lower()
    return TOKEN_RE.findall(s)

In [ ]:
import duckdb
import os, shutil

os.makedirs('/home/ec2-user/SageMaker/tmp', exist_ok=True)

con = duckdb.connect('/home/ec2-user/SageMaker/blocking.duckdb')
con.execute("PRAGMA memory_limit='22GB'")   # ~70% of 32GB
# Reconnecting to the same file in the same kernel reuses the open database, and DuckDB refuses to
# change its spill folder once it has spilled. So only set it if it isn't already this folder.
if con.execute("SELECT current_setting('temp_directory')").fetchone()[0] != '/home/ec2-user/SageMaker/tmp':
    con.execute("PRAGMA temp_directory='/home/ec2-user/SageMaker/tmp'")

free_gb = shutil.disk_usage('/home/ec2-user/SageMaker').free / 1e9
print(f"Free disk: {free_gb:.1f} GB")

In [ ]:
con.execute("PRAGMA memory_limit='16GB'")   # down from 22GB, more room for pandas-side work

In [ ]:
import gc
import time
import numpy as np

CHUNK_SIZE = 300_000  # rows per chunk; tune down further if still tight

def load_tokens_chunked(df, field_name, col, chunk_size=CHUNK_SIZE):
    n_chunks = int(np.ceil(len(df) / chunk_size))
    for i in range(n_chunks):
        chunk = df.iloc[i*chunk_size : (i+1)*chunk_size]
        tmp = pd.DataFrame({
            'entity_id': chunk['entity_id'].values,
            'token': chunk[col].apply(tokenize).values
        })
        tmp = tmp.explode('token').dropna(subset=['token'])
        tmp['field'] = field_name
        con.append('tokens', tmp[['entity_id', 'field', 'token']])
        del chunk, tmp
        gc.collect()
        if i % 5 == 0:
            print(f"  {field_name}/{col}: chunk {i+1}/{n_chunks} done")

con.execute("CREATE OR REPLACE TABLE tokens (entity_id VARCHAR, field VARCHAR, token VARCHAR)")

t0 = time.time()
for df in [s2, s3]:
    load_tokens_chunked(df, 'N', 'business_name')
    load_tokens_chunked(df, 'A', 'business_address')
print(f"tokens load took {time.time()-t0:.1f}s")
print(con.execute("SELECT COUNT(*) FROM tokens").fetchone())

In [ ]:
MAX_TOKENS = 5  # tokens per field considered for pairing

con.execute("""
CREATE OR REPLACE TABLE token_freq AS
SELECT token, COUNT(DISTINCT entity_id) AS doc_freq FROM tokens GROUP BY token
""")

con.execute(f"""
CREATE OR REPLACE TABLE tokens_top AS
SELECT entity_id, field, token, doc_freq FROM (
    SELECT t.entity_id, t.field, t.token, f.doc_freq,
           ROW_NUMBER() OVER (PARTITION BY t.entity_id, t.field ORDER BY f.doc_freq ASC, t.token) AS rnk
    FROM tokens t JOIN token_freq f USING (token)
) WHERE rnk <= {MAX_TOKENS}
""")
print("tokens_top rows:", con.execute("SELECT COUNT(*) FROM tokens_top").fetchone())

In [ ]:
t0 = time.time()
con.execute("""
CREATE OR REPLACE TABLE pair_freq AS
SELECT a.field, a.token AS token_a, b.token AS token_b,
       COUNT(DISTINCT a.entity_id) AS doc_freq
FROM tokens_top a JOIN tokens_top b
  ON a.entity_id = b.entity_id AND a.field = b.field AND a.token < b.token
GROUP BY a.field, a.token, b.token
""")
print(f"pair_freq build took {(time.time()-t0)/60:.1f} min")

print("distinct pairs:", con.execute("SELECT COUNT(*) FROM pair_freq").fetchone())
print(con.execute("""
    SELECT field, quantile_cont(doc_freq,0.5), quantile_cont(doc_freq,0.99),
           quantile_cont(doc_freq,0.999), max(doc_freq)
    FROM pair_freq GROUP BY field
""").fetchall())

free_gb = shutil.disk_usage('/home/ec2-user/SageMaker').free / 1e9
print(f"Free disk remaining: {free_gb:.1f} GB")

In [ ]:
for cap in [500, 1000, 2000, 3000, 5000, 10000]:
    result = con.execute(f"""
        SELECT field,
               COUNT(*) AS n_pairs_over_cap,
               SUM(doc_freq) AS total_entity_associations_over_cap
        FROM pair_freq
        WHERE doc_freq > {cap}
        GROUP BY field
    """).fetchall()
    total_pairs = con.execute("SELECT COUNT(*) FROM pair_freq").fetchone()[0]
    print(f"cap={cap}: {result}  (out of {total_pairs} total distinct pairs)")

In [ ]:
PAIR_CAP = 2000

con.execute(f"""
CREATE OR REPLACE TABLE pair_index AS
SELECT field, token_a, token_b, doc_freq
FROM pair_freq
WHERE doc_freq <= {PAIR_CAP}
""")
print("pair_index pairs kept:", con.execute("SELECT COUNT(*) FROM pair_index").fetchone()[0])

# (field, token_a, token_b) -> S2/S3 candidate entity_id   [the join target]
con.execute("""
CREATE OR REPLACE TABLE pair_postings AS
SELECT p.field, p.token_a, p.token_b, a.entity_id
FROM pair_index p
JOIN tokens_top a ON a.field = p.field AND a.token = p.token_a
JOIN tokens_top b ON b.field = p.field AND b.token = p.token_b AND b.entity_id = a.entity_id
""")
print("pair_postings rows:", con.execute("SELECT COUNT(*) FROM pair_postings").fetchone()[0])

In [ ]:
del s2, s3, s2_idx, s3_idx
gc.collect()
print("freed s2/s3 pandas frames")

In [ ]:
# id_to_country built straight from the source TSVs so it survives the del above.
con.execute(f"""
CREATE OR REPLACE TABLE cand_country AS
SELECT CAST(entity_id AS VARCHAR) AS entity_id, country
FROM read_csv_auto('{BASE}/train/train_source2.tsv', delim='\t', header=true)
UNION ALL
SELECT CAST(entity_id AS VARCHAR) AS entity_id, country
FROM read_csv_auto('{BASE}/train/train_source3.tsv', delim='\t', header=true)
""")
print("cand_country rows:", con.execute("SELECT COUNT(*) FROM cand_country").fetchone()[0])

In [ ]:
# S1 tokens -> tokens_top_s1 -> pairs_s1, mirroring the S2+S3 pipeline exactly.
con.execute("CREATE OR REPLACE TABLE tokens_s1 (entity_id VARCHAR, field VARCHAR, token VARCHAR)")

def load_tokens_chunked_s1(df, field_name, col, chunk_size=CHUNK_SIZE):
    n_chunks = int(np.ceil(len(df) / chunk_size))
    for i in range(n_chunks):
        chunk = df.iloc[i*chunk_size:(i+1)*chunk_size]
        tmp = pd.DataFrame({
            'entity_id': chunk['entity_id'].astype(str).values,
            'token': chunk[col].apply(tokenize).values
        })
        tmp = tmp.explode('token').dropna(subset=['token'])
        tmp['field'] = field_name
        con.append('tokens_s1', tmp[['entity_id', 'field', 'token']])
        del chunk, tmp
        gc.collect()
        if i % 3 == 0:
            print(f"  {field_name}: chunk {i+1}/{n_chunks}")

t0 = time.time()
load_tokens_chunked_s1(s1, 'N', 'business_name')
load_tokens_chunked_s1(s1, 'A', 'business_address')
print(f"tokens_s1 load: {time.time()-t0:.1f}s, rows:",
      con.execute("SELECT COUNT(*) FROM tokens_s1").fetchone()[0])

In [ ]:
# Rank S1 tokens by the SAME token_freq (S2+S3 doc-freq). Tokens absent from S2/S3
# get doc_freq via LEFT JOIN -> a huge sentinel so they sort LAST (never chosen as
# a rarest key, since they can never match a candidate anyway).
con.execute(f"""
CREATE OR REPLACE TABLE tokens_top_s1 AS
SELECT entity_id, field, token, doc_freq FROM (
    SELECT t.entity_id, t.field, t.token,
           COALESCE(f.doc_freq, 999999999) AS doc_freq,
           ROW_NUMBER() OVER (PARTITION BY t.entity_id, t.field
                              ORDER BY COALESCE(f.doc_freq, 999999999) ASC, t.token) AS rnk
    FROM tokens_s1 t LEFT JOIN token_freq f USING (token)
) WHERE rnk <= {MAX_TOKENS}
""")
print("tokens_top_s1 rows:", con.execute("SELECT COUNT(*) FROM tokens_top_s1").fetchone()[0])

con.execute("""
CREATE OR REPLACE TABLE pairs_s1 AS
SELECT a.entity_id, a.field, a.token AS token_a, b.token AS token_b
FROM tokens_top_s1 a JOIN tokens_top_s1 b
  ON a.entity_id = b.entity_id AND a.field = b.field AND a.token < b.token
""")
print("pairs_s1 rows:", con.execute("SELECT COUNT(*) FROM pairs_s1").fetchone()[0])

In [ ]:
free_gb = shutil.disk_usage('/home/ec2-user/SageMaker').free / 1e9
print(f"Free disk: {free_gb:.1f} GB")

In [ ]:
# ABANDONED APPROACH, kept for the record. Blocking on every token *pair* over the full train set
# produced so many (S1, candidate) rows that the final DISTINCT spilled past 80 GB of temp disk and
# failed with OutOfMemoryException. Nothing later in this notebook or in pipeline_v5.py reads
# `candidates` / `candidates_dedup`; the lesson (filter and sample before a big join, then work in
# hash buckets) is what the next cells and v5 do instead. Set this to True only to reproduce the failure.
RUN_ABANDONED_PAIR_BLOCKING = False

if RUN_ABANDONED_PAIR_BLOCKING:
    con.execute("SET preserve_insertion_order=false")
    con.execute("PRAGMA max_temp_directory_size='80GiB'")  # leave ~10GB headroom on a 93GB disk

    t0 = time.time()
    con.execute("CREATE OR REPLACE TABLE candidates (s1_entity_id VARCHAR, cand_entity_id VARCHAR)")

    for f in ['N', 'A']:
        con.execute(f"""
        INSERT INTO candidates
        SELECT p.entity_id AS s1_entity_id, pp.entity_id AS cand_entity_id
        FROM pairs_s1 p
        JOIN pair_postings pp
          ON pp.field = p.field AND pp.token_a = p.token_a AND pp.token_b = p.token_b
        WHERE p.field = '{f}'
        GROUP BY p.entity_id, pp.entity_id
        """)
        print(f"  field={f} done, {time.time()-t0:.0f}s elapsed")

    print(f"candidates build: {(time.time()-t0)/60:.1f} min")

    # dedupe across the two field-passes (an (s1,cand) pair matched on BOTH name and addr pairs)
    con.execute("CREATE OR REPLACE TABLE candidates_dedup AS SELECT DISTINCT s1_entity_id, cand_entity_id FROM candidates")
    print("candidate rows (deduped, pre country filter):", con.execute("SELECT COUNT(*) FROM candidates_dedup").fetchone()[0])
else:
    print("Skipped: token-pair blocking was abandoned (its DISTINCT spilled >80 GB). See the comment above.")

In [ ]:
import pandas as pd, numpy as np, os, gc, time, shutil
import unicodedata, regex
import duckdb

DATA_ROOT = "/home/ec2-user/SageMaker/data"
BASE = f"{DATA_ROOT}/dataset"

TOKEN_RE = regex.compile(r'[\p{L}\p{M}\p{N}]+')
def tokenize(s):
    if pd.isnull(s):
        return []
    s = unicodedata.normalize('NFC', str(s)).lower()
    return TOKEN_RE.findall(s)

con = duckdb.connect('/home/ec2-user/SageMaker/blocking.duckdb')
con.execute("PRAGMA memory_limit='16GB'")
# Reconnecting to the same file in the same kernel reuses the open database, and DuckDB refuses to
# change its spill folder once it has spilled. So only set it if it isn't already this folder.
if con.execute("SELECT current_setting('temp_directory')").fetchone()[0] != '/home/ec2-user/SageMaker/tmp':
    con.execute("PRAGMA temp_directory='/home/ec2-user/SageMaker/tmp'")
con.execute("SET preserve_insertion_order=false")

s1 = pd.read_csv(f"{BASE}/train/train_source1.tsv", sep="\t")
gt = pd.read_csv(f"{BASE}/train/train_ground_truth.tsv", sep="\t")
gt['matched_entity_ids'] = gt['matched_entity_ids'].fillna('')
gt['match_list'] = gt['matched_entity_ids'].apply(lambda x: [i.strip() for i in x.split(',') if i.strip() != ''])
gt['n_matches'] = gt['match_list'].apply(len)
s1_idx = s1.set_index('entity_id')

tables = con.execute("SHOW TABLES").fetchall()
print("Existing tables:", [t[0] for t in tables])
for tname in [t[0] for t in tables]:
    n = con.execute(f"SELECT COUNT(*) FROM {tname}").fetchone()[0]
    print(f"  {tname}: {n:,} rows")

free_gb = shutil.disk_usage('/home/ec2-user/SageMaker').free / 1e9
print(f"free disk: {free_gb:.1f} GB")

In [ ]:
n_s1_done = con.execute("SELECT COUNT(DISTINCT s1_entity_id) FROM candidates_capped").fetchone()[0]
mean_per_entity = con.execute("SELECT AVG(cnt) FROM (SELECT s1_entity_id, COUNT(*) AS cnt FROM candidates_capped GROUP BY s1_entity_id)").fetchone()[0]
print(f"S1 entities with candidates so far: {n_s1_done:,} / 2,206,821")
print(f"mean candidates/entity so far: {mean_per_entity:.1f}")

In [ ]:
done_ids = con.execute("SELECT DISTINCT s1_entity_id FROM candidates_capped").fetchdf()['s1_entity_id'].tolist()
all_ids = con.execute("SELECT DISTINCT entity_id FROM pairs_s1").fetchdf()['entity_id'].tolist()
remaining_ids = list(set(all_ids) - set(done_ids))
print(f"remaining: {len(remaining_ids):,} entities")

BATCH_SIZE = 50_000
TOP_K = 200
n_batches = int(np.ceil(len(remaining_ids) / BATCH_SIZE))
print(f"{n_batches} batches remaining")

t0 = time.time()
for i in range(n_batches):
    batch_ids = remaining_ids[i*BATCH_SIZE:(i+1)*BATCH_SIZE]
    con.execute("CREATE OR REPLACE TEMP TABLE batch_ids (entity_id VARCHAR)")
    con.executemany("INSERT INTO batch_ids VALUES (?)", [(x,) for x in batch_ids])
    con.execute(f"""
    INSERT INTO candidates_capped
    SELECT s1_entity_id, cand_entity_id, best_doc_freq FROM (
        SELECT
            p.entity_id AS s1_entity_id, pp.entity_id AS cand_entity_id,
            MIN(pi.doc_freq) AS best_doc_freq,
            ROW_NUMBER() OVER (PARTITION BY p.entity_id ORDER BY MIN(pi.doc_freq) ASC) AS rnk
        FROM pairs_s1 p
        JOIN batch_ids b ON b.entity_id = p.entity_id
        JOIN pair_postings pp ON pp.field = p.field AND pp.token_a = p.token_a AND pp.token_b = p.token_b
        JOIN pair_index pi ON pi.field = p.field AND pi.token_a = p.token_a AND pi.token_b = p.token_b
        GROUP BY p.entity_id, pp.entity_id
        QUALIFY rnk <= {TOP_K}
    )
    """)
    elapsed = time.time() - t0
    free_gb = shutil.disk_usage('/home/ec2-user/SageMaker').free / 1e9
    print(f"  batch {i+1}/{n_batches} done, {elapsed:.0f}s elapsed, free disk: {free_gb:.1f}GB")

print(f"resume build: {(time.time()-t0)/60:.1f} min")
n = con.execute("SELECT COUNT(*) FROM candidates_capped").fetchone()[0]
n_s1 = con.execute("SELECT COUNT(DISTINCT s1_entity_id) FROM candidates_capped").fetchone()[0]
print(f"total rows: {n:,}, S1 covered: {n_s1:,}/2,206,821, mean/entity: {n/max(n_s1,1):.1f}")

In [ ]:
missing = con.execute("""
    SELECT DISTINCT p.entity_id FROM pairs_s1 p
    WHERE p.entity_id NOT IN (SELECT DISTINCT s1_entity_id FROM candidates_capped)
""").fetchdf()
print(missing)
# also check entities with NO pairs at all (not even attempted)
all_s1 = set(con.execute("SELECT entity_id FROM s1").fetchdf()['entity_id'].astype(str))
have_pairs = set(con.execute("SELECT DISTINCT entity_id FROM pairs_s1").fetchdf()['entity_id'])
no_pairs_at_all = all_s1 - have_pairs
print(f"S1 entities with zero pairs (not even in pairs_s1): {len(no_pairs_at_all):,}")

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE candidates_final AS
SELECT c.s1_entity_id, c.cand_entity_id, c.best_doc_freq
FROM candidates_capped c
JOIN s1_country sc ON sc.entity_id = c.s1_entity_id
JOIN cand_country cc ON cc.entity_id = c.cand_entity_id AND cc.country = sc.country
""")
n = con.execute("SELECT COUNT(*) FROM candidates_final").fetchone()[0]
n_s1 = con.execute("SELECT COUNT(DISTINCT s1_entity_id) FROM candidates_final").fetchone()[0]
print(f"candidates_final rows: {n:,}, S1 covered: {n_s1:,}, mean/entity: {n/max(n_s1,1):.1f}")

In [ ]:
val_sample = gt.sample(5000, random_state=2).copy()
val_ids = set(val_sample['source1_entity_id'].astype(str))

cand_map = con.execute(f"""
    SELECT s1_entity_id, LIST(cand_entity_id) AS cands
    FROM candidates_final
    WHERE s1_entity_id IN ({','.join(f"'{i}'" for i in val_ids)})
    GROUP BY s1_entity_id
""").fetchdf()
cand_lookup = dict(zip(cand_map['s1_entity_id'], cand_map['cands']))

full_recall_count = 0
sizes = []
for _, row in val_sample.iterrows():
    sid = str(row['source1_entity_id'])
    true_matches = set(row['match_list'])
    got = set(cand_lookup.get(sid, []))
    sizes.append(len(got))
    if true_matches.issubset(got) or len(true_matches) == 0:
        full_recall_count += 1

print(f"full-recall rate: {full_recall_count/5000*100:.2f}%  (handoff v3/v_hybrid baseline: 84.05%/84.26%)")
print(f"mean candidate size: {np.mean(sizes):.1f}, p50: {np.median(sizes):.0f}, p99: {np.percentile(sizes,99):.0f}")

In [ ]:
# Quick check: for the 5K val sample, how many additional candidates appear
# in the NEXT 100 ranked slots (rnk 201-300) that a higher TOP_K would capture?
# Re-run the same join scoped to the val sample only, with TOP_K=500, to see
# the recall curve without rebuilding the full 2.2M-entity table yet.
val_ids_sql = ','.join(f"'{i}'" for i in val_ids)

con.execute(f"""
CREATE OR REPLACE TABLE val_probe AS
SELECT s1_entity_id, cand_entity_id, rnk FROM (
    SELECT
        p.entity_id AS s1_entity_id, pp.entity_id AS cand_entity_id,
        MIN(pi.doc_freq) AS best_doc_freq,
        ROW_NUMBER() OVER (PARTITION BY p.entity_id ORDER BY MIN(pi.doc_freq) ASC) AS rnk
    FROM pairs_s1 p
    JOIN pair_postings pp ON pp.field = p.field AND pp.token_a = p.token_a AND pp.token_b = p.token_b
    JOIN pair_index pi ON pi.field = p.field AND pi.token_a = p.token_a AND pi.token_b = p.token_b
    WHERE p.entity_id IN ({val_ids_sql})
    GROUP BY p.entity_id, pp.entity_id
    QUALIFY rnk <= 1000
)
""")

for k in [200, 300, 500, 1000]:
    probe_map = con.execute(f"""
        SELECT s1_entity_id, LIST(cand_entity_id) AS cands
        FROM val_probe WHERE rnk <= {k}
        GROUP BY s1_entity_id
    """).fetchdf()
    lookup = dict(zip(probe_map['s1_entity_id'], probe_map['cands']))
    # country filter applied in Python for this quick probe
    s1c = dict(zip(con.execute("SELECT entity_id, country FROM s1_country").fetchdf()['entity_id'],
                    con.execute("SELECT entity_id, country FROM s1_country").fetchdf()['country']))
    cc = con.execute("SELECT entity_id, country FROM cand_country").fetchdf()
    cc_map = dict(zip(cc['entity_id'], cc['country']))

    hits = 0
    for _, row in val_sample.iterrows():
        sid = str(row['source1_entity_id'])
        true_matches = set(row['match_list'])
        got_raw = set(lookup.get(sid, []))
        got = {c for c in got_raw if cc_map.get(c) == s1c.get(sid)}
        if true_matches.issubset(got) or len(true_matches) == 0:
            hits += 1
    print(f"TOP_K={k}: full-recall = {hits/5000*100:.2f}%")

In [ ]:
# Compare candidates_final (validated at 90.14%) against val_probe (showing 50.14%)
# for the SAME small set of entities, to find the actual discrepancy.
sample_ids = list(val_ids)[:5]

for sid in sample_ids:
    final_cands = set(con.execute(
        f"SELECT cand_entity_id FROM candidates_final WHERE s1_entity_id = '{sid}'"
    ).fetchdf()['cand_entity_id'])
    probe_cands = set(con.execute(
        f"SELECT cand_entity_id FROM val_probe WHERE s1_entity_id = '{sid}' AND rnk <= 200"
    ).fetchdf()['cand_entity_id'])
    print(f"{sid}: candidates_final={len(final_cands)}, val_probe(rnk<=200)={len(probe_cands)}, "
          f"overlap={len(final_cands & probe_cands)}")

In [ ]:
con.execute("DROP TABLE IF EXISTS candidates_capped")
con.execute("DROP TABLE IF EXISTS candidates_final")
con.execute("DROP TABLE IF EXISTS val_probe")
con.execute("CREATE OR REPLACE TABLE candidates_capped (s1_entity_id VARCHAR, cand_entity_id VARCHAR, best_doc_freq BIGINT)")

BATCH_SIZE = 50_000
TOP_K = 200
all_ids = con.execute("SELECT DISTINCT entity_id FROM pairs_s1").fetchdf()['entity_id'].tolist()
n_batches = int(np.ceil(len(all_ids) / BATCH_SIZE))
print(f"{len(all_ids):,} entities, {n_batches} batches")

t0 = time.time()
for i in range(n_batches):
    batch_ids = all_ids[i*BATCH_SIZE:(i+1)*BATCH_SIZE]
    con.execute("CREATE OR REPLACE TEMP TABLE batch_ids (entity_id VARCHAR)")
    con.executemany("INSERT INTO batch_ids VALUES (?)", [(x,) for x in batch_ids])
    con.execute(f"""
    INSERT INTO candidates_capped
    SELECT s1_entity_id, cand_entity_id, best_doc_freq FROM (
        SELECT
            p.entity_id AS s1_entity_id, pp.entity_id AS cand_entity_id,
            MIN(pi.doc_freq) AS best_doc_freq,
            ROW_NUMBER() OVER (
                PARTITION BY p.entity_id
                ORDER BY MIN(pi.doc_freq) ASC, pp.entity_id ASC
            ) AS rnk
        FROM pairs_s1 p
        JOIN batch_ids b ON b.entity_id = p.entity_id
        JOIN pair_postings pp ON pp.field = p.field AND pp.token_a = p.token_a AND pp.token_b = p.token_b
        JOIN pair_index pi ON pi.field = p.field AND pi.token_a = p.token_a AND pi.token_b = p.token_b
        GROUP BY p.entity_id, pp.entity_id
        QUALIFY rnk <= {TOP_K}
    )
    """)
    if (i+1) % 5 == 0 or i == n_batches-1:
        print(f"  batch {i+1}/{n_batches}, {time.time()-t0:.0f}s elapsed")

print(f"build: {(time.time()-t0)/60:.1f} min")
n = con.execute("SELECT COUNT(*) FROM candidates_capped").fetchone()[0]
n_s1 = con.execute("SELECT COUNT(DISTINCT s1_entity_id) FROM candidates_capped").fetchone()[0]
print(f"rows: {n:,}, S1 covered: {n_s1:,}, mean/entity: {n/max(n_s1,1):.1f}")

In [ ]:
con.execute("""
CREATE OR REPLACE TABLE candidates_final AS
SELECT c.s1_entity_id, c.cand_entity_id, c.best_doc_freq
FROM candidates_capped c
JOIN s1_country sc ON sc.entity_id = c.s1_entity_id
JOIN cand_country cc ON cc.entity_id = c.cand_entity_id AND cc.country = sc.country
""")
print("candidates_final rows:", con.execute("SELECT COUNT(*) FROM candidates_final").fetchone()[0])

In [ ]:
val_sample = gt.sample(5000, random_state=2).copy()
val_ids = set(val_sample['source1_entity_id'].astype(str))
cand_map = con.execute(f"""
    SELECT s1_entity_id, LIST(cand_entity_id) AS cands FROM candidates_final
    WHERE s1_entity_id IN ({','.join(f"'{i}'" for i in val_ids)})
    GROUP BY s1_entity_id
""").fetchdf()
cand_lookup = dict(zip(cand_map['s1_entity_id'], cand_map['cands']))

full_recall_count = 0
sizes = []
for _, row in val_sample.iterrows():
    sid = str(row['source1_entity_id'])
    true_matches = set(row['match_list'])
    got = set(cand_lookup.get(sid, []))
    sizes.append(len(got))
    if true_matches.issubset(got) or len(true_matches) == 0:
        full_recall_count += 1
print(f"full-recall rate: {full_recall_count/5000*100:.2f}%")
print(f"mean candidate size: {np.mean(sizes):.1f}")

In [ ]:
t0 = time.time()
con.execute("""
CREATE OR REPLACE TABLE token_counts_s1 AS
SELECT entity_id, field, COUNT(DISTINCT token) AS n_tok
FROM tokens_s1 GROUP BY entity_id, field
""")
print(f"token_counts_s1: {time.time()-t0:.1f}s, rows: {con.execute('SELECT COUNT(*) FROM token_counts_s1').fetchone()[0]:,}")

t0 = time.time()
con.execute("""
CREATE OR REPLACE TABLE token_counts_cand AS
SELECT entity_id, field, COUNT(DISTINCT token) AS n_tok
FROM tokens GROUP BY entity_id, field
""")
print(f"token_counts_cand: {time.time()-t0:.1f}s, rows: {con.execute('SELECT COUNT(*) FROM token_counts_cand').fetchone()[0]:,}")

In [ ]:
# ABANDONED PROBES, kept for the record. These three cells joined the sample's S1 tokens to the
# full 117M-row `tokens` table on the bare token value. A common token such as 'road' or 'pvt' has
# hundreds of thousands of postings, so a 5K-entity sample still exploded into billions of rows and
# spilled past the 80 GB temp-disk cap (OutOfMemoryException). The fix, a document-frequency cap on
# the tokens allowed into the join, comes a few cells later and rebuilds `token_intersect_sample`,
# so nothing downstream needs these. Set this to True only to reproduce the failure.
RUN_UNCAPPED_TOKEN_PROBES = False

if RUN_UNCAPPED_TOKEN_PROBES:
    t0 = time.time()
    con.execute("""
    CREATE OR REPLACE TABLE token_intersect_sample AS
    SELECT s.entity_id AS s1_entity_id, t.entity_id AS cand_entity_id, s.field,
           COUNT(DISTINCT s.token) AS n_shared
    FROM tokens_s1 s
    JOIN s1_sample ss ON ss.entity_id = s.entity_id
    JOIN tokens t ON t.field = s.field AND t.token = s.token
    JOIN candidates_final c ON c.s1_entity_id = s.entity_id AND c.cand_entity_id = t.entity_id
    GROUP BY s.entity_id, t.entity_id, s.field
    """)
    elapsed = time.time() - t0
    n = con.execute("SELECT COUNT(*) FROM token_intersect_sample").fetchone()[0]
    print(f"token_intersect_sample: {elapsed:.1f}s, rows: {n:,}")
else:
    print("Skipped: uncapped token-value join (abandoned, see comment above RUN_UNCAPPED_TOKEN_PROBES)")


In [ ]:
# Part of the abandoned uncapped-join probes (see RUN_UNCAPPED_TOKEN_PROBES two cells up).
if RUN_UNCAPPED_TOKEN_PROBES:
    con.execute("SET preserve_insertion_order=false")  # safe to re-run if already set

    t0 = time.time()
    n_pairs = con.execute("""
        SELECT COUNT(*) FROM candidates_final c JOIN s1_sample ss ON ss.entity_id = c.s1_entity_id
    """).fetchone()[0]
    print(f"step 1 (candidates_final x sample count): {time.time()-t0:.1f}s, pairs: {n_pairs:,}")

    t0 = time.time()
    n_s1toks = con.execute("""
        SELECT COUNT(*) FROM tokens_s1 s JOIN s1_sample ss ON ss.entity_id = s.entity_id
    """).fetchone()[0]
    print(f"step 2 (tokens_s1 x sample count): {time.time()-t0:.1f}s, rows: {n_s1toks:,}")

    t0 = time.time()
    con.execute("""
    CREATE OR REPLACE TABLE probe_step3 AS
    SELECT s.entity_id AS s1_entity_id, s.field, s.token, COUNT(*) AS n_matches
    FROM tokens_s1 s
    JOIN s1_sample ss ON ss.entity_id = s.entity_id
    JOIN tokens t ON t.field = s.field AND t.token = s.token
    GROUP BY s.entity_id, s.field, s.token
    """)
    print(f"step 3 (token-value join, sample s1 tokens x full tokens): {time.time()-t0:.1f}s")
    n3 = con.execute("SELECT COUNT(*) FROM probe_step3").fetchone()[0]
    print(f"  rows: {n3:,}")
    print(con.execute("SELECT * FROM probe_step3 ORDER BY n_matches DESC LIMIT 10").fetchdf())
else:
    print("Skipped: uncapped token-value join (abandoned, see comment above RUN_UNCAPPED_TOKEN_PROBES)")


In [ ]:
# Part of the abandoned uncapped-join probes (see RUN_UNCAPPED_TOKEN_PROBES two cells up).
if RUN_UNCAPPED_TOKEN_PROBES:
    # Correct order: start from candidates_final (bounded), THEN pull in tokens,
    # joined per (s1_entity_id, cand_entity_id) pair already fixed by candidates_final --
    # never join tokens_s1 <-> tokens directly on bare token value.
    t0 = time.time()
    con.execute("""
    CREATE OR REPLACE TABLE token_intersect_sample AS
    SELECT c.s1_entity_id, c.cand_entity_id, s.field, COUNT(DISTINCT s.token) AS n_shared
    FROM candidates_final c
    JOIN s1_sample ss ON ss.entity_id = c.s1_entity_id
    JOIN tokens_s1 s ON s.entity_id = c.s1_entity_id
    JOIN tokens t ON t.entity_id = c.cand_entity_id AND t.field = s.field AND t.token = s.token
    GROUP BY c.s1_entity_id, c.cand_entity_id, s.field
    """)
    elapsed = time.time() - t0
    n = con.execute("SELECT COUNT(*) FROM token_intersect_sample").fetchone()[0]
    print(f"token_intersect_sample: {elapsed:.1f}s, rows: {n:,}")
else:
    print("Skipped: uncapped token-value join (abandoned, see comment above RUN_UNCAPPED_TOKEN_PROBES)")


In [ ]:
import pandas as pd, numpy as np, os, gc, time, shutil
import unicodedata, regex
import duckdb

DATA_ROOT = "/home/ec2-user/SageMaker/data"
BASE = f"{DATA_ROOT}/dataset"
TOKEN_RE = regex.compile(r'[\p{L}\p{M}\p{N}]+')
def tokenize(s):
    if pd.isnull(s):
        return []
    s = unicodedata.normalize('NFC', str(s)).lower()
    return TOKEN_RE.findall(s)

con = duckdb.connect('/home/ec2-user/SageMaker/blocking.duckdb')
con.execute("PRAGMA memory_limit='16GB'")
# Reconnecting to the same file in the same kernel reuses the open database, and DuckDB refuses to
# change its spill folder once it has spilled. So only set it if it isn't already this folder.
if con.execute("SELECT current_setting('temp_directory')").fetchone()[0] != '/home/ec2-user/SageMaker/tmp':
    con.execute("PRAGMA temp_directory='/home/ec2-user/SageMaker/tmp'")
con.execute("SET preserve_insertion_order=false")

s1 = pd.read_csv(f"{BASE}/train/train_source1.tsv", sep="\t")
gt = pd.read_csv(f"{BASE}/train/train_ground_truth.tsv", sep="\t")
gt['matched_entity_ids'] = gt['matched_entity_ids'].fillna('')
gt['match_list'] = gt['matched_entity_ids'].apply(lambda x: [i.strip() for i in x.split(',') if i.strip() != ''])

print("Tables:", [t[0] for t in con.execute("SHOW TABLES").fetchall()])
print("free disk:", shutil.disk_usage('/home/ec2-user/SageMaker').free / 1e9, "GB")

In [ ]:
t0 = time.time()
n_pairs = con.execute("""
    SELECT COUNT(*) FROM candidates_final c JOIN s1_sample ss ON ss.entity_id = c.s1_entity_id
""").fetchone()[0]
print(f"step 1: {time.time()-t0:.1f}s, pairs: {n_pairs:,}")

t0 = time.time()
n_s1toks = con.execute("""
    SELECT COUNT(*) FROM tokens_s1 s JOIN s1_sample ss ON ss.entity_id = s.entity_id
""").fetchone()[0]
print(f"step 2: {time.time()-t0:.1f}s, rows: {n_s1toks:,}")

In [ ]:
# A: how many DISTINCT tokens does the 5K sample even touch? (should be fast, small)
t0 = time.time()
n = con.execute("""
    SELECT COUNT(DISTINCT s.token) FROM tokens_s1 s JOIN s1_sample ss ON ss.entity_id = s.entity_id
""").fetchone()[0]
print(f"A: {time.time()-t0:.1f}s -- distinct tokens in sample: {n:,}")

In [ ]:
# B: of those, how many are "common" (high doc_freq) vs "rare"? This tells us the
# flood's actual size BEFORE attempting any join to the 117M-row `tokens` table.
t0 = time.time()
result = con.execute("""
    SELECT
        SUM(CASE WHEN f.doc_freq > 50000 THEN 1 ELSE 0 END) AS common_tokens,
        SUM(CASE WHEN f.doc_freq <= 50000 THEN 1 ELSE 0 END) AS rare_tokens,
        SUM(CASE WHEN f.doc_freq > 50000 THEN f.doc_freq ELSE 0 END) AS common_total_postings
    FROM (SELECT DISTINCT s.token FROM tokens_s1 s JOIN s1_sample ss ON ss.entity_id = s.entity_id) dt
    JOIN token_freq f ON f.token = dt.token
""").fetchone()
print(f"B: {time.time()-t0:.1f}s -- common_tokens={result[0]}, rare_tokens={result[1]}, common_total_postings={result[2]:,}")

In [ ]:
# C: NOW the actual intersect join, but with the token_freq cap applied,
# so it should skip the flood entirely and finish fast.
t0 = time.time()
con.execute("""
CREATE OR REPLACE TABLE token_intersect_sample AS
SELECT c.s1_entity_id, c.cand_entity_id, s.field, COUNT(DISTINCT s.token) AS n_shared
FROM candidates_final c
JOIN s1_sample ss ON ss.entity_id = c.s1_entity_id
JOIN tokens_s1 s ON s.entity_id = c.s1_entity_id
JOIN token_freq f ON f.token = s.token AND f.doc_freq <= 50000
JOIN tokens t ON t.entity_id = c.cand_entity_id AND t.field = s.field AND t.token = s.token
GROUP BY c.s1_entity_id, c.cand_entity_id, s.field
""")
print(f"C: {time.time()-t0:.1f}s")
n = con.execute("SELECT COUNT(*) FROM token_intersect_sample").fetchone()[0]
print(f"rows: {n:,}")

In [ ]:
JACCARD_TOKEN_FREQ_CAP = 5000  # tighter cap -- these are near-universal tokens for Jaccard purposes anyway

t0 = time.time()
result = con.execute(f"""
    SELECT
        SUM(CASE WHEN f.doc_freq > {JACCARD_TOKEN_FREQ_CAP} THEN 1 ELSE 0 END) AS common_tokens,
        SUM(CASE WHEN f.doc_freq <= {JACCARD_TOKEN_FREQ_CAP} THEN 1 ELSE 0 END) AS rare_tokens
    FROM (SELECT DISTINCT s.token FROM tokens_s1 s JOIN s1_sample ss ON ss.entity_id = s.entity_id) dt
    JOIN token_freq f ON f.token = dt.token
""").fetchone()
print(f"at cap={JACCARD_TOKEN_FREQ_CAP}: common={result[0]}, rare={result[1]}")

t0 = time.time()
con.execute(f"""
CREATE OR REPLACE TABLE token_intersect_sample2 AS
SELECT c.s1_entity_id, c.cand_entity_id, s.field, COUNT(DISTINCT s.token) AS n_shared
FROM candidates_final c
JOIN s1_sample ss ON ss.entity_id = c.s1_entity_id
JOIN tokens_s1 s ON s.entity_id = c.s1_entity_id
JOIN token_freq f ON f.token = s.token AND f.doc_freq <= {JACCARD_TOKEN_FREQ_CAP}
JOIN tokens t ON t.entity_id = c.cand_entity_id AND t.field = s.field AND t.token = s.token
GROUP BY c.s1_entity_id, c.cand_entity_id, s.field
""")
print(f"C (cap={JACCARD_TOKEN_FREQ_CAP}): {time.time()-t0:.1f}s")
print("rows:", con.execute("SELECT COUNT(*) FROM token_intersect_sample2").fetchone()[0])

In [ ]:
t0 = time.time()
con.execute(f"""
CREATE OR REPLACE TABLE token_intersect_sample50k AS
SELECT c.s1_entity_id, c.cand_entity_id, s.field, COUNT(DISTINCT s.token) AS n_shared
FROM candidates_final c
JOIN s1_sample50k ss ON ss.entity_id = c.s1_entity_id
JOIN tokens_s1 s ON s.entity_id = c.s1_entity_id
JOIN token_freq f ON f.token = s.token AND f.doc_freq <= 5000
JOIN tokens t ON t.entity_id = c.cand_entity_id AND t.field = s.field AND t.token = s.token
GROUP BY c.s1_entity_id, c.cand_entity_id, s.field
""")
elapsed = time.time() - t0
n = con.execute("SELECT COUNT(*) FROM token_intersect_sample50k").fetchone()[0]
print(f"50K sample: {elapsed:.1f}s, rows: {n:,}")
print(f"projected full-scale (linear): {elapsed * 2206821/50000 / 60:.1f} min")

In [ ]:
JACCARD_TOKEN_FREQ_CAP = 5000
BATCH_SIZE = 50_000

con.execute("DROP TABLE IF EXISTS token_intersect")
con.execute("CREATE OR REPLACE TABLE token_intersect (s1_entity_id VARCHAR, cand_entity_id VARCHAR, field VARCHAR, n_shared BIGINT)")

all_ids = con.execute("SELECT DISTINCT entity_id FROM pairs_s1").fetchdf()['entity_id'].tolist()
n_batches = int(np.ceil(len(all_ids) / BATCH_SIZE))
print(f"{len(all_ids):,} entities, {n_batches} batches")

t0 = time.time()
for i in range(n_batches):
    batch_ids = all_ids[i*BATCH_SIZE:(i+1)*BATCH_SIZE]
    con.execute("CREATE OR REPLACE TEMP TABLE batch_ids (entity_id VARCHAR)")
    con.executemany("INSERT INTO batch_ids VALUES (?)", [(x,) for x in batch_ids])

    con.execute(f"""
    INSERT INTO token_intersect
    SELECT c.s1_entity_id, c.cand_entity_id, s.field, COUNT(DISTINCT s.token) AS n_shared
    FROM candidates_final c
    JOIN batch_ids b ON b.entity_id = c.s1_entity_id
    JOIN tokens_s1 s ON s.entity_id = c.s1_entity_id
    JOIN token_freq f ON f.token = s.token AND f.doc_freq <= {JACCARD_TOKEN_FREQ_CAP}
    JOIN tokens t ON t.entity_id = c.cand_entity_id AND t.field = s.field AND t.token = s.token
    GROUP BY c.s1_entity_id, c.cand_entity_id, s.field
    """)
    elapsed = time.time() - t0
    print(f"  batch {i+1}/{n_batches}, {elapsed:.0f}s elapsed")

print(f"total build: {(time.time()-t0)/60:.1f} min")
n = con.execute("SELECT COUNT(*) FROM token_intersect").fetchone()[0]
print(f"rows: {n:,}")

In [ ]:
t0 = time.time()
con.execute(f"""
CREATE OR REPLACE TABLE jaccard_features AS
SELECT
    ti.s1_entity_id, ti.cand_entity_id, ti.field, ti.n_shared,
    cs.n_tok AS s1_n_tok, cc.n_tok AS cand_n_tok,
    ti.n_shared * 1.0 / GREATEST(cs.n_tok + cc.n_tok - ti.n_shared, 1) AS jaccard
FROM token_intersect ti
JOIN token_counts_s1 cs ON cs.entity_id = ti.s1_entity_id AND cs.field = ti.field
JOIN token_counts_cand cc ON cc.entity_id = ti.cand_entity_id AND cc.field = ti.field
""")
print(f"jaccard_features build: {(time.time()-t0)/60:.1f} min")
n = con.execute("SELECT COUNT(*) FROM jaccard_features").fetchone()[0]
print(f"rows: {n:,}")

In [ ]:
val_sample = gt.sample(5000, random_state=2).copy()
true_pairs = []
for _, row in val_sample.iterrows():
    sid = str(row['source1_entity_id'])
    for mid in row['match_list']:
        true_pairs.append((sid, mid))
true_pairs_df = pd.DataFrame(true_pairs, columns=['s1_entity_id', 'cand_entity_id'])
print(f"true match pairs in sample: {len(true_pairs_df):,}")
con.register('true_pairs_view', true_pairs_df)

for field in ['N', 'A']:
    stats = con.execute(f"""
        SELECT AVG(jf.jaccard) AS mean_jaccard,
               SUM(CASE WHEN jf.jaccard = 0 THEN 1 ELSE 0 END) AS zero_count,
               COUNT(*) AS total,
               SUM(CASE WHEN jf.s1_entity_id IS NULL THEN 1 ELSE 0 END) AS no_row_found
        FROM true_pairs_view tp
        LEFT JOIN jaccard_features jf
          ON jf.s1_entity_id = tp.s1_entity_id AND jf.cand_entity_id = tp.cand_entity_id AND jf.field = '{field}'
    """).fetchone()
    print(f"field={field}: mean_jaccard={stats[0]}, zero_count={stats[1]}, total={stats[2]}, no_row_found(no shared rare token)={stats[3]}")

In [ ]:
con.execute("DROP TABLE IF EXISTS jaccard_features_v2")
con.execute("CREATE OR REPLACE TABLE jaccard_features_v2 (s1_entity_id VARCHAR, cand_entity_id VARCHAR, field VARCHAR, n_shared BIGINT, s1_n_tok BIGINT, cand_n_tok BIGINT, jaccard DOUBLE, has_shared_rare_token BOOLEAN)")

all_ids = con.execute("SELECT DISTINCT entity_id FROM pairs_s1").fetchdf()['entity_id'].tolist()
BATCH_SIZE = 50_000
n_batches = int(np.ceil(len(all_ids) / BATCH_SIZE))
print(f"{len(all_ids):,} entities, {n_batches} batches")

t0 = time.time()
for i in range(n_batches):
    batch_ids = all_ids[i*BATCH_SIZE:(i+1)*BATCH_SIZE]
    con.execute("CREATE OR REPLACE TEMP TABLE batch_ids (entity_id VARCHAR)")
    con.executemany("INSERT INTO batch_ids VALUES (?)", [(x,) for x in batch_ids])

    con.execute(f"""
    INSERT INTO jaccard_features_v2
    SELECT
        c.s1_entity_id, c.cand_entity_id, allf.field,
        COALESCE(ti.n_shared, 0),
        COALESCE(cs.n_tok, 0),
        COALESCE(cc.n_tok, 0),
        COALESCE(ti.n_shared, 0) * 1.0 /
            GREATEST(COALESCE(cs.n_tok,0) + COALESCE(cc.n_tok,0) - COALESCE(ti.n_shared,0), 1),
        (ti.n_shared IS NOT NULL)
    FROM candidates_final c
    JOIN batch_ids b ON b.entity_id = c.s1_entity_id
    CROSS JOIN (SELECT 'N' AS field UNION SELECT 'A') allf
    LEFT JOIN token_intersect ti
      ON ti.s1_entity_id = c.s1_entity_id AND ti.cand_entity_id = c.cand_entity_id AND ti.field = allf.field
    LEFT JOIN token_counts_s1 cs ON cs.entity_id = c.s1_entity_id AND cs.field = allf.field
    LEFT JOIN token_counts_cand cc ON cc.entity_id = c.cand_entity_id AND cc.field = allf.field
    """)
    print(f"  batch {i+1}/{n_batches}, {time.time()-t0:.0f}s elapsed")

print(f"total build: {(time.time()-t0)/60:.1f} min")
n = con.execute("SELECT COUNT(*) FROM jaccard_features_v2").fetchone()[0]
print(f"rows: {n:,} (expected ~{con.execute('SELECT COUNT(*)*2 FROM candidates_final').fetchone()[0]:,})")

In [ ]:
# Alternative shape -- avoid CROSS JOIN, use UNION ALL of two independent per-field
# LEFT JOIN chains instead. Same semantics, likely friendlier to the query planner.
def build_batch(field):
    return f"""
    SELECT
        c.s1_entity_id, c.cand_entity_id, '{field}' AS field,
        COALESCE(ti.n_shared, 0),
        COALESCE(cs.n_tok, 0),
        COALESCE(cc.n_tok, 0),
        COALESCE(ti.n_shared, 0) * 1.0 /
            GREATEST(COALESCE(cs.n_tok,0) + COALESCE(cc.n_tok,0) - COALESCE(ti.n_shared,0), 1),
        (ti.n_shared IS NOT NULL)
    FROM candidates_final c
    JOIN batch_ids b ON b.entity_id = c.s1_entity_id
    LEFT JOIN token_intersect ti
      ON ti.s1_entity_id = c.s1_entity_id AND ti.cand_entity_id = c.cand_entity_id AND ti.field = '{field}'
    LEFT JOIN token_counts_s1 cs ON cs.entity_id = c.s1_entity_id AND cs.field = '{field}'
    LEFT JOIN token_counts_cand cc ON cc.entity_id = c.cand_entity_id AND cc.field = '{field}'
    """

In [ ]:
con.execute("DROP TABLE IF EXISTS jaccard_features_v2")
con.execute("CREATE OR REPLACE TABLE jaccard_features_v2 (s1_entity_id VARCHAR, cand_entity_id VARCHAR, field VARCHAR, n_shared BIGINT, s1_n_tok BIGINT, cand_n_tok BIGINT, jaccard DOUBLE, has_shared_rare_token BOOLEAN)")

all_ids = con.execute("SELECT DISTINCT entity_id FROM pairs_s1").fetchdf()['entity_id'].tolist()
BATCH_SIZE = 50_000
n_batches = int(np.ceil(len(all_ids) / BATCH_SIZE))
print(f"{len(all_ids):,} entities, {n_batches} batches")

t0 = time.time()
for i in range(n_batches):
    batch_ids = all_ids[i*BATCH_SIZE:(i+1)*BATCH_SIZE]
    con.execute("CREATE OR REPLACE TEMP TABLE batch_ids (entity_id VARCHAR)")
    con.executemany("INSERT INTO batch_ids VALUES (?)", [(x,) for x in batch_ids])

    con.execute("""
    INSERT INTO jaccard_features_v2
    SELECT
        c.s1_entity_id, c.cand_entity_id, 'N' AS field,
        COALESCE(ti.n_shared, 0),
        COALESCE(cs.n_tok, 0),
        COALESCE(cc.n_tok, 0),
        COALESCE(ti.n_shared, 0) * 1.0 /
            GREATEST(COALESCE(cs.n_tok,0) + COALESCE(cc.n_tok,0) - COALESCE(ti.n_shared,0), 1),
        (ti.n_shared IS NOT NULL)
    FROM candidates_final c
    JOIN batch_ids b ON b.entity_id = c.s1_entity_id
    LEFT JOIN token_intersect ti
      ON ti.s1_entity_id = c.s1_entity_id AND ti.cand_entity_id = c.cand_entity_id AND ti.field = 'N'
    LEFT JOIN token_counts_s1 cs ON cs.entity_id = c.s1_entity_id AND cs.field = 'N'
    LEFT JOIN token_counts_cand cc ON cc.entity_id = c.cand_entity_id AND cc.field = 'N'
    UNION ALL
    SELECT
        c.s1_entity_id, c.cand_entity_id, 'A' AS field,
        COALESCE(ti.n_shared, 0),
        COALESCE(cs.n_tok, 0),
        COALESCE(cc.n_tok, 0),
        COALESCE(ti.n_shared, 0) * 1.0 /
            GREATEST(COALESCE(cs.n_tok,0) + COALESCE(cc.n_tok,0) - COALESCE(ti.n_shared,0), 1),
        (ti.n_shared IS NOT NULL)
    FROM candidates_final c
    JOIN batch_ids b ON b.entity_id = c.s1_entity_id
    LEFT JOIN token_intersect ti
      ON ti.s1_entity_id = c.s1_entity_id AND ti.cand_entity_id = c.cand_entity_id AND ti.field = 'A'
    LEFT JOIN token_counts_s1 cs ON cs.entity_id = c.s1_entity_id AND cs.field = 'A'
    LEFT JOIN token_counts_cand cc ON cc.entity_id = c.cand_entity_id AND cc.field = 'A'
    """)
    print(f"  batch {i+1}/{n_batches}, {time.time()-t0:.0f}s elapsed")

print(f"total build: {(time.time()-t0)/60:.1f} min")
n = con.execute("SELECT COUNT(*) FROM jaccard_features_v2").fetchone()[0]
print(f"rows: {n:,} (expected ~{con.execute('SELECT COUNT(*)*2 FROM candidates_final').fetchone()[0]:,})")

In [ ]:
val_sample = gt.sample(5000, random_state=2).copy()
true_pairs = []
for _, row in val_sample.iterrows():
    sid = str(row['source1_entity_id'])
    for mid in row['match_list']:
        true_pairs.append((sid, mid))
true_pairs_df = pd.DataFrame(true_pairs, columns=['s1_entity_id', 'cand_entity_id'])
print(f"true match pairs in sample: {len(true_pairs_df):,}")
con.register('true_pairs_view', true_pairs_df)

for field in ['N', 'A']:
    stats = con.execute(f"""
        SELECT AVG(jf.jaccard), SUM(CASE WHEN jf.jaccard=0 THEN 1 ELSE 0 END), COUNT(*),
               SUM(CASE WHEN jf.has_shared_rare_token THEN 0 ELSE 1 END) AS no_qualifying_token
        FROM true_pairs_view tp
        JOIN jaccard_features_v2 jf
          ON jf.s1_entity_id = tp.s1_entity_id AND jf.cand_entity_id = tp.cand_entity_id AND jf.field = '{field}'
    """).fetchone()
    print(f"field={field}: mean_jaccard={stats[0]:.4f}, zero_count={stats[1]}, total={stats[2]}, no_qualifying_token={stats[3]}")

In [ ]:
missing_check = con.execute("""
    SELECT tp.s1_entity_id, tp.cand_entity_id
    FROM true_pairs_view tp
    LEFT JOIN candidates_final cf
      ON cf.s1_entity_id = tp.s1_entity_id AND cf.cand_entity_id = tp.cand_entity_id
    WHERE cf.s1_entity_id IS NULL
""").fetchdf()
print(f"true pairs missing from candidates_final entirely: {len(missing_check)}")

In [ ]:
false_sample = con.execute("""
    SELECT jf.jaccard FROM jaccard_features_v2 jf
    ANTI JOIN true_pairs_view tp
      ON tp.s1_entity_id = jf.s1_entity_id AND tp.cand_entity_id = jf.cand_entity_id
    WHERE jf.field = 'N' AND jf.s1_entity_id IN (SELECT entity_id FROM s1_sample)
    USING SAMPLE 20000
""").fetchdf()
true_name = con.execute("""
    SELECT jf.jaccard FROM jaccard_features_v2 jf
    JOIN true_pairs_view tp
      ON tp.s1_entity_id = jf.s1_entity_id AND tp.cand_entity_id = jf.cand_entity_id
    WHERE jf.field = 'N'
""").fetchdf()
print(f"TRUE matches name jaccard: mean={true_name['jaccard'].mean():.4f}, median={true_name['jaccard'].median():.4f}")
print(f"FALSE candidates name jaccard: mean={false_sample['jaccard'].mean():.4f}, median={false_sample['jaccard'].median():.4f}")

In [ ]:
# exact match approximated via existing token infrastructure: jaccard==1 AND
# both entities have the same non-zero token count in that field (avoids
# re-reading raw TSVs or building a new large string-lookup table).
t0 = time.time()
con.execute("""
CREATE OR REPLACE TABLE exact_match_features AS
SELECT s1_entity_id, cand_entity_id, field,
       (jaccard = 1.0 AND s1_n_tok > 0 AND s1_n_tok = cand_n_tok) AS is_exact_match
FROM jaccard_features_v2
""")
print(f"exact_match_features: {(time.time()-t0)/60:.1f} min")
print("rows:", con.execute("SELECT COUNT(*) FROM exact_match_features").fetchone()[0])

In [ ]:
import importlib
try:
    import rapidfuzz
    print("rapidfuzz available:", rapidfuzz.__version__)
except ImportError:
    print("NOT installed -- run: !pip install rapidfuzz")

In [ ]:
t0 = time.time()
con.execute(f"""
CREATE OR REPLACE TABLE raw_strings_cand AS
SELECT CAST(entity_id AS VARCHAR) AS entity_id, business_name, business_address
FROM read_csv_auto('{BASE}/train/train_source2.tsv', delim='\t', header=true)
UNION ALL
SELECT CAST(entity_id AS VARCHAR) AS entity_id, business_name, business_address
FROM read_csv_auto('{BASE}/train/train_source3.tsv', delim='\t', header=true)
""")
print(f"raw_strings_cand: {time.time()-t0:.1f}s, rows: {con.execute('SELECT COUNT(*) FROM raw_strings_cand').fetchone()[0]:,}")

con.register('s1_pd', s1)
con.execute("""
CREATE OR REPLACE TABLE raw_strings_s1 AS
SELECT CAST(entity_id AS VARCHAR) AS entity_id, business_name, business_address FROM s1_pd
""")
print("raw_strings_s1 rows:", con.execute("SELECT COUNT(*) FROM raw_strings_s1").fetchone()[0])

In [ ]:
import rapidfuzz
from rapidfuzz.distance import JaroWinkler

t0 = time.time()
pair_strings_sample = con.execute("""
    SELECT c.s1_entity_id, c.cand_entity_id,
           s1r.business_name AS s1_name, cr.business_name AS cand_name,
           s1r.business_address AS s1_addr, cr.business_address AS cand_addr
    FROM candidates_final c
    JOIN s1_sample ss ON ss.entity_id = c.s1_entity_id
    JOIN raw_strings_s1 s1r ON s1r.entity_id = c.s1_entity_id
    JOIN raw_strings_cand cr ON cr.entity_id = c.cand_entity_id
""").fetchdf()
print(f"pulled: {time.time()-t0:.1f}s, rows: {len(pair_strings_sample):,}")

t0 = time.time()
pair_strings_sample['name_jw'] = pair_strings_sample.apply(
    lambda r: JaroWinkler.normalized_similarity(str(r['s1_name']), str(r['cand_name'])), axis=1
)
pair_strings_sample['addr_jw'] = pair_strings_sample.apply(
    lambda r: JaroWinkler.normalized_similarity(str(r['s1_addr']) if pd.notnull(r['s1_addr']) else '',
                                                  str(r['cand_addr']) if pd.notnull(r['cand_addr']) else ''),
    axis=1
)
print(f"jaro-winkler compute: {time.time()-t0:.1f}s")
print(pair_strings_sample[['s1_entity_id','cand_entity_id','name_jw','addr_jw']].head(10))
print(f"mean name_jw: {pair_strings_sample['name_jw'].mean():.4f}, mean addr_jw: {pair_strings_sample['addr_jw'].mean():.4f}")

In [ ]:
from rapidfuzz.distance import JaroWinkler

t0 = time.time()
names_s1 = pair_strings_sample['s1_name'].astype(str).tolist()
names_cand = pair_strings_sample['cand_name'].astype(str).tolist()
addrs_s1 = pair_strings_sample['s1_addr'].fillna('').astype(str).tolist()
addrs_cand = pair_strings_sample['cand_addr'].fillna('').astype(str).tolist()

name_jw = [JaroWinkler.normalized_similarity(a, b) for a, b in zip(names_s1, names_cand)]
addr_jw = [JaroWinkler.normalized_similarity(a, b) for a, b in zip(addrs_s1, addrs_cand)]
print(f"list-comprehension pairwise: {time.time()-t0:.1f}s")

In [ ]:
con.execute("DROP TABLE IF EXISTS jw_features")
con.execute("CREATE OR REPLACE TABLE jw_features (s1_entity_id VARCHAR, cand_entity_id VARCHAR, name_jw DOUBLE, addr_jw DOUBLE)")

all_ids = con.execute("SELECT DISTINCT entity_id FROM pairs_s1").fetchdf()['entity_id'].tolist()
BATCH_SIZE = 50_000
n_batches = int(np.ceil(len(all_ids) / BATCH_SIZE))
print(f"{len(all_ids):,} entities, {n_batches} batches")

t0 = time.time()
for i in range(n_batches):
    batch_ids = all_ids[i*BATCH_SIZE:(i+1)*BATCH_SIZE]
    con.execute("CREATE OR REPLACE TEMP TABLE batch_ids (entity_id VARCHAR)")
    con.executemany("INSERT INTO batch_ids VALUES (?)", [(x,) for x in batch_ids])

    batch_df = con.execute("""
        SELECT c.s1_entity_id, c.cand_entity_id,
               s1r.business_name AS s1_name, cr.business_name AS cand_name,
               s1r.business_address AS s1_addr, cr.business_address AS cand_addr
        FROM candidates_final c
        JOIN batch_ids b ON b.entity_id = c.s1_entity_id
        JOIN raw_strings_s1 s1r ON s1r.entity_id = c.s1_entity_id
        JOIN raw_strings_cand cr ON cr.entity_id = c.cand_entity_id
    """).fetchdf()

    names_s1 = batch_df['s1_name'].fillna('').astype(str).tolist()
    names_cand = batch_df['cand_name'].fillna('').astype(str).tolist()
    addrs_s1 = batch_df['s1_addr'].fillna('').astype(str).tolist()
    addrs_cand = batch_df['cand_addr'].fillna('').astype(str).tolist()

    batch_df['name_jw'] = [JaroWinkler.normalized_similarity(a, b) for a, b in zip(names_s1, names_cand)]
    batch_df['addr_jw'] = [JaroWinkler.normalized_similarity(a, b) for a, b in zip(addrs_s1, addrs_cand)]

    con.register('batch_result', batch_df[['s1_entity_id', 'cand_entity_id', 'name_jw', 'addr_jw']])
    con.execute("INSERT INTO jw_features SELECT * FROM batch_result")

    print(f"  batch {i+1}/{n_batches}, {time.time()-t0:.0f}s elapsed")

print(f"total build: {(time.time()-t0)/60:.1f} min")
n = con.execute("SELECT COUNT(*) FROM jw_features").fetchone()[0]
print(f"rows: {n:,} (expected ~{con.execute('SELECT COUNT(*) FROM candidates_final').fetchone()[0]:,})")

In [ ]:
import pandas as pd, numpy as np, os, gc, time, shutil
import unicodedata, regex
import duckdb
from rapidfuzz.distance import JaroWinkler

DATA_ROOT = "/home/ec2-user/SageMaker/data"
BASE = f"{DATA_ROOT}/dataset"
TOKEN_RE = regex.compile(r'[\p{L}\p{M}\p{N}]+')
def tokenize(s):
    if pd.isnull(s):
        return []
    s = unicodedata.normalize('NFC', str(s)).lower()
    return TOKEN_RE.findall(s)

con = duckdb.connect('/home/ec2-user/SageMaker/blocking.duckdb')
con.execute("PRAGMA memory_limit='16GB'")
# Reconnecting to the same file in the same kernel reuses the open database, and DuckDB refuses to
# change its spill folder once it has spilled. So only set it if it isn't already this folder.
if con.execute("SELECT current_setting('temp_directory')").fetchone()[0] != '/home/ec2-user/SageMaker/tmp':
    con.execute("PRAGMA temp_directory='/home/ec2-user/SageMaker/tmp'")
con.execute("SET preserve_insertion_order=false")

s1 = pd.read_csv(f"{BASE}/train/train_source1.tsv", sep="\t")
gt = pd.read_csv(f"{BASE}/train/train_ground_truth.tsv", sep="\t")
gt['matched_entity_ids'] = gt['matched_entity_ids'].fillna('')
gt['match_list'] = gt['matched_entity_ids'].apply(lambda x: [i.strip() for i in x.split(',') if i.strip() != ''])

print("Tables:", [t[0] for t in con.execute("SHOW TABLES").fetchall()])
free_gb = shutil.disk_usage('/home/ec2-user/SageMaker').free / 1e9
print(f"free disk: {free_gb:.1f} GB")

In [ ]:
n_jw = con.execute("SELECT COUNT(*) FROM jw_features").fetchone()[0]
print(f"jw_features existing rows: {n_jw:,}")

con.execute("PRAGMA memory_limit='10GB'")  # down from 16GB -- leave more headroom for pandas/rapidfuzz
free_gb = shutil.disk_usage('/home/ec2-user/SageMaker').free / 1e9
print(f"free disk: {free_gb:.1f} GB")

In [ ]:
con.execute("PRAGMA memory_limit='10GB'")

done_ids = con.execute("SELECT DISTINCT s1_entity_id FROM jw_features").fetchdf()['s1_entity_id'].tolist()
all_ids = con.execute("SELECT DISTINCT entity_id FROM pairs_s1").fetchdf()['entity_id'].tolist()
remaining_ids = list(set(all_ids) - set(done_ids))
print(f"done: {len(done_ids):,}, remaining: {len(remaining_ids):,}")

BATCH_SIZE = 10_000
n_test_batches = 3  # still just a monitored test first, not the full remaining set

t0 = time.time()
for i in range(n_test_batches):
    batch_ids = remaining_ids[i*BATCH_SIZE:(i+1)*BATCH_SIZE]
    batch_ids_df = pd.DataFrame({'entity_id': batch_ids})
    con.register('batch_ids_view', batch_ids_df)

    batch_df = con.execute("""
        SELECT c.s1_entity_id, c.cand_entity_id,
               s1r.business_name AS s1_name, cr.business_name AS cand_name,
               s1r.business_address AS s1_addr, cr.business_address AS cand_addr
        FROM candidates_final c
        JOIN batch_ids_view b ON b.entity_id = c.s1_entity_id
        JOIN raw_strings_s1 s1r ON s1r.entity_id = c.s1_entity_id
        JOIN raw_strings_cand cr ON cr.entity_id = c.cand_entity_id
    """).fetchdf()

    names_s1 = batch_df['s1_name'].fillna('').astype(str).tolist()
    names_cand = batch_df['cand_name'].fillna('').astype(str).tolist()
    addrs_s1 = batch_df['s1_addr'].fillna('').astype(str).tolist()
    addrs_cand = batch_df['cand_addr'].fillna('').astype(str).tolist()

    name_jw = [JaroWinkler.normalized_similarity(a, b) for a, b in zip(names_s1, names_cand)]
    addr_jw = [JaroWinkler.normalized_similarity(a, b) for a, b in zip(addrs_s1, addrs_cand)]

    result_df = pd.DataFrame({
        's1_entity_id': batch_df['s1_entity_id'].values,
        'cand_entity_id': batch_df['cand_entity_id'].values,
        'name_jw': name_jw,
        'addr_jw': addr_jw,
    })
    con.register('batch_result', result_df)
    con.execute("INSERT INTO jw_features SELECT * FROM batch_result")
    con.unregister('batch_result')
    con.unregister('batch_ids_view')
    del batch_df, result_df, names_s1, names_cand, addrs_s1, addrs_cand, name_jw, addr_jw
    gc.collect()

    print(f"  test batch {i+1}/{n_test_batches}, {time.time()-t0:.0f}s elapsed")

print(f"test build: {(time.time()-t0)/60:.1f} min")
print("rows so far:", con.execute("SELECT COUNT(*) FROM jw_features").fetchone()[0])

In [ ]:
BATCH_SIZE = 10_000
n_batches = int(np.ceil(len(remaining_ids) / BATCH_SIZE))
print(f"{len(remaining_ids):,} remaining entities, {n_batches} batches")

t0 = time.time()
for i in range(n_batches):
    batch_ids = remaining_ids[i*BATCH_SIZE:(i+1)*BATCH_SIZE]
    batch_ids_df = pd.DataFrame({'entity_id': batch_ids})
    con.register('batch_ids_view', batch_ids_df)

    batch_df = con.execute("""
        SELECT c.s1_entity_id, c.cand_entity_id,
               s1r.business_name AS s1_name, cr.business_name AS cand_name,
               s1r.business_address AS s1_addr, cr.business_address AS cand_addr
        FROM candidates_final c
        JOIN batch_ids_view b ON b.entity_id = c.s1_entity_id
        JOIN raw_strings_s1 s1r ON s1r.entity_id = c.s1_entity_id
        JOIN raw_strings_cand cr ON cr.entity_id = c.cand_entity_id
    """).fetchdf()

    names_s1 = batch_df['s1_name'].fillna('').astype(str).tolist()
    names_cand = batch_df['cand_name'].fillna('').astype(str).tolist()
    addrs_s1 = batch_df['s1_addr'].fillna('').astype(str).tolist()
    addrs_cand = batch_df['cand_addr'].fillna('').astype(str).tolist()

    name_jw = [JaroWinkler.normalized_similarity(a, b) for a, b in zip(names_s1, names_cand)]
    addr_jw = [JaroWinkler.normalized_similarity(a, b) for a, b in zip(addrs_s1, addrs_cand)]

    result_df = pd.DataFrame({
        's1_entity_id': batch_df['s1_entity_id'].values,
        'cand_entity_id': batch_df['cand_entity_id'].values,
        'name_jw': name_jw,
        'addr_jw': addr_jw,
    })
    con.register('batch_result', result_df)
    con.execute("INSERT INTO jw_features SELECT * FROM batch_result")
    con.unregister('batch_result')
    con.unregister('batch_ids_view')
    del batch_df, result_df, names_s1, names_cand, addrs_s1, addrs_cand, name_jw, addr_jw
    gc.collect()

    if (i+1) % 10 == 0 or i == n_batches-1:
        print(f"  batch {i+1}/{n_batches}, {time.time()-t0:.0f}s elapsed")

print(f"build: {(time.time()-t0)/60:.1f} min")
print("total rows:", con.execute("SELECT COUNT(*) FROM jw_features").fetchone()[0])

In [ ]:
n_total = con.execute("SELECT COUNT(*) FROM jw_features").fetchone()[0]
n_distinct_pairs = con.execute("SELECT COUNT(DISTINCT (s1_entity_id, cand_entity_id)) FROM jw_features").fetchone()[0]
n_expected = con.execute("SELECT COUNT(*) FROM candidates_final").fetchone()[0]
print(f"jw_features rows: {n_total:,}")
print(f"distinct pairs: {n_distinct_pairs:,}")
print(f"expected (candidates_final): {n_expected:,}")

In [ ]:
# 1. Dedupe jw_features -- keep one row per pair (first occurrence is fine, values should be identical)
t0 = time.time()
con.execute("""
CREATE OR REPLACE TABLE jw_features_dedup AS
SELECT s1_entity_id, cand_entity_id, ANY_VALUE(name_jw) AS name_jw, ANY_VALUE(addr_jw) AS addr_jw
FROM jw_features
GROUP BY s1_entity_id, cand_entity_id
""")
print(f"dedup: {(time.time()-t0)/60:.1f} min")
n_dedup = con.execute("SELECT COUNT(*) FROM jw_features_dedup").fetchone()[0]
print(f"jw_features_dedup rows: {n_dedup:,} (expected 411,043,867)")

# swap it in once confirmed
if n_dedup == 411043867:
    con.execute("DROP TABLE jw_features")
    con.execute("ALTER TABLE jw_features_dedup RENAME TO jw_features")
    print("jw_features replaced with deduped version")

In [ ]:
val_sample = gt.sample(5000, random_state=2).copy()
true_pairs = []
for _, row in val_sample.iterrows():
    sid = str(row['source1_entity_id'])
    for mid in row['match_list']:
        true_pairs.append((sid, mid))
true_pairs_df = pd.DataFrame(true_pairs, columns=['s1_entity_id', 'cand_entity_id'])
print(f"true match pairs in sample: {len(true_pairs_df):,}")
con.register('true_pairs_view', true_pairs_df)

true_jw = con.execute("""
    SELECT jf.name_jw, jf.addr_jw FROM jw_features jf
    JOIN true_pairs_view tp
      ON tp.s1_entity_id = jf.s1_entity_id AND tp.cand_entity_id = jf.cand_entity_id
""").fetchdf()
false_jw = con.execute("""
    SELECT jf.name_jw, jf.addr_jw FROM jw_features jf
    ANTI JOIN true_pairs_view tp
      ON tp.s1_entity_id = jf.s1_entity_id AND tp.cand_entity_id = jf.cand_entity_id
    WHERE jf.s1_entity_id IN (SELECT entity_id FROM s1_sample)
    USING SAMPLE 20000
""").fetchdf()
print(f"TRUE  name_jw: mean={true_jw['name_jw'].mean():.4f}, addr_jw: mean={true_jw['addr_jw'].mean():.4f}")
print(f"FALSE name_jw: mean={false_jw['name_jw'].mean():.4f}, addr_jw: mean={false_jw['addr_jw'].mean():.4f}")

In [ ]:
# Build full ground-truth pair set (not just the 5K validation sample) for labeling
t0 = time.time()
all_true_pairs = []
for _, row in gt.iterrows():
    sid = str(row['source1_entity_id'])
    for mid in row['match_list']:
        all_true_pairs.append((sid, mid))
all_true_pairs_df = pd.DataFrame(all_true_pairs, columns=['s1_entity_id', 'cand_entity_id'])
print(f"total true pairs (full ground truth): {len(all_true_pairs_df):,}, build: {time.time()-t0:.1f}s")
con.register('all_true_pairs_view', all_true_pairs_df)
con.execute("CREATE OR REPLACE TABLE all_true_pairs AS SELECT * FROM all_true_pairs_view")
print("all_true_pairs table rows:", con.execute("SELECT COUNT(*) FROM all_true_pairs").fetchone()[0])

In [ ]:
con.execute("SET memory_limit='8GB'")  # stay under the ceiling that just OOM'd, leave clear host headroom

# Split jaccard/exact_match into per-field tables ONCE (small, fast, avoids repeated field-filtering per join)
t0 = time.time()
con.execute("CREATE OR REPLACE TABLE jaccard_n AS SELECT s1_entity_id, cand_entity_id, jaccard AS name_jaccard, has_shared_rare_token AS name_has_shared_rare_token FROM jaccard_features_v2 WHERE field='N'")
con.execute("CREATE OR REPLACE TABLE jaccard_a AS SELECT s1_entity_id, cand_entity_id, jaccard AS addr_jaccard, has_shared_rare_token AS addr_has_shared_rare_token FROM jaccard_features_v2 WHERE field='A'")
con.execute("CREATE OR REPLACE TABLE exact_n AS SELECT s1_entity_id, cand_entity_id, is_exact_match AS name_exact_approx FROM exact_match_features WHERE field='N'")
con.execute("CREATE OR REPLACE TABLE exact_a AS SELECT s1_entity_id, cand_entity_id, is_exact_match AS addr_exact_approx FROM exact_match_features WHERE field='A'")
print(f"split tables: {(time.time()-t0)/60:.1f} min")
for t in ['jaccard_n','jaccard_a','exact_n','exact_a']:
    print(t, con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0])

In [ ]:
# Drop the failed test table, free memory
con.execute("DROP TABLE IF EXISTS training_matrix_test")
con.execute("SET memory_limit='8GB'")

# Test: filter EVERY table (both feature tables AND all_true_pairs) to the batch's
# s1_entity_id BEFORE joining -- this bounds every hash table to batch size, not 411M.
t0 = time.time()
con.execute("""
CREATE OR REPLACE TABLE training_matrix_test AS
WITH batch_jn AS (SELECT jn.* FROM jaccard_n jn JOIN s1_sample ss ON ss.entity_id = jn.s1_entity_id),
     batch_ja AS (SELECT ja.* FROM jaccard_a ja JOIN s1_sample ss ON ss.entity_id = ja.s1_entity_id),
     batch_en AS (SELECT en.* FROM exact_n en JOIN s1_sample ss ON ss.entity_id = en.s1_entity_id),
     batch_ea AS (SELECT ea.* FROM exact_a ea JOIN s1_sample ss ON ss.entity_id = ea.s1_entity_id),
     batch_jw AS (SELECT jw.* FROM jw_features jw JOIN s1_sample ss ON ss.entity_id = jw.s1_entity_id),
     batch_tp AS (SELECT tp.* FROM all_true_pairs tp JOIN s1_sample ss ON ss.entity_id = tp.s1_entity_id)
SELECT
    c.s1_entity_id, c.cand_entity_id,
    jn.name_jaccard, jn.name_has_shared_rare_token,
    ja.addr_jaccard, ja.addr_has_shared_rare_token,
    en.name_exact_approx, ea.addr_exact_approx,
    jw.name_jw, jw.addr_jw,
    (tp.s1_entity_id IS NOT NULL) AS is_match
FROM candidates_final c
JOIN s1_sample ss ON ss.entity_id = c.s1_entity_id
LEFT JOIN batch_jn jn ON jn.s1_entity_id = c.s1_entity_id AND jn.cand_entity_id = c.cand_entity_id
LEFT JOIN batch_ja ja ON ja.s1_entity_id = c.s1_entity_id AND ja.cand_entity_id = c.cand_entity_id
LEFT JOIN batch_en en ON en.s1_entity_id = c.s1_entity_id AND en.cand_entity_id = c.cand_entity_id
LEFT JOIN batch_ea ea ON ea.s1_entity_id = c.s1_entity_id AND ea.cand_entity_id = c.cand_entity_id
LEFT JOIN batch_jw jw ON jw.s1_entity_id = c.s1_entity_id AND jw.cand_entity_id = c.cand_entity_id
LEFT JOIN batch_tp tp ON tp.s1_entity_id = c.s1_entity_id AND tp.cand_entity_id = c.cand_entity_id
""")
print(f"test matrix build: {time.time()-t0:.1f}s")
n = con.execute("SELECT COUNT(*) FROM training_matrix_test").fetchone()[0]
n_pos = con.execute("SELECT COUNT(*) FROM training_matrix_test WHERE is_match").fetchone()[0]
print(f"rows: {n:,}, positive: {n_pos:,} ({n_pos/n*100:.3f}%)")

In [ ]:
con.execute("DROP TABLE IF EXISTS training_matrix_test")

con.execute("DROP TABLE IF EXISTS training_matrix")
con.execute("""CREATE OR REPLACE TABLE training_matrix (
    s1_entity_id VARCHAR, cand_entity_id VARCHAR,
    name_jaccard DOUBLE, name_has_shared_rare_token BOOLEAN,
    addr_jaccard DOUBLE, addr_has_shared_rare_token BOOLEAN,
    name_exact_approx BOOLEAN, addr_exact_approx BOOLEAN,
    name_jw DOUBLE, addr_jw DOUBLE,
    is_match BOOLEAN
)""")

all_ids = con.execute("SELECT DISTINCT entity_id FROM pairs_s1").fetchdf()['entity_id'].tolist()
BATCH_SIZE = 20_000
n_batches = int(np.ceil(len(all_ids) / BATCH_SIZE))
print(f"{len(all_ids):,} entities, {n_batches} batches")

t0 = time.time()
for i in range(n_batches):
    batch_ids = all_ids[i*BATCH_SIZE:(i+1)*BATCH_SIZE]
    batch_ids_df = pd.DataFrame({'entity_id': batch_ids})
    con.register('batch_ids_view', batch_ids_df)

    con.execute("""
    INSERT INTO training_matrix
    WITH batch_jn AS (SELECT jn.* FROM jaccard_n jn JOIN batch_ids_view b ON b.entity_id = jn.s1_entity_id),
         batch_ja AS (SELECT ja.* FROM jaccard_a ja JOIN batch_ids_view b ON b.entity_id = ja.s1_entity_id),
         batch_en AS (SELECT en.* FROM exact_n en JOIN batch_ids_view b ON b.entity_id = en.s1_entity_id),
         batch_ea AS (SELECT ea.* FROM exact_a ea JOIN batch_ids_view b ON b.entity_id = ea.s1_entity_id),
         batch_jw AS (SELECT jw.* FROM jw_features jw JOIN batch_ids_view b ON b.entity_id = jw.s1_entity_id),
         batch_tp AS (SELECT tp.* FROM all_true_pairs tp JOIN batch_ids_view b ON b.entity_id = tp.s1_entity_id)
    SELECT
        c.s1_entity_id, c.cand_entity_id,
        jn.name_jaccard, jn.name_has_shared_rare_token,
        ja.addr_jaccard, ja.addr_has_shared_rare_token,
        en.name_exact_approx, ea.addr_exact_approx,
        jw.name_jw, jw.addr_jw,
        (tp.s1_entity_id IS NOT NULL) AS is_match
    FROM candidates_final c
    JOIN batch_ids_view b ON b.entity_id = c.s1_entity_id
    LEFT JOIN batch_jn jn ON jn.s1_entity_id = c.s1_entity_id AND jn.cand_entity_id = c.cand_entity_id
    LEFT JOIN batch_ja ja ON ja.s1_entity_id = c.s1_entity_id AND ja.cand_entity_id = c.cand_entity_id
    LEFT JOIN batch_en en ON en.s1_entity_id = c.s1_entity_id AND en.cand_entity_id = c.cand_entity_id
    LEFT JOIN batch_ea ea ON ea.s1_entity_id = c.s1_entity_id AND ea.cand_entity_id = c.cand_entity_id
    LEFT JOIN batch_jw jw ON jw.s1_entity_id = c.s1_entity_id AND jw.cand_entity_id = c.cand_entity_id
    LEFT JOIN batch_tp tp ON tp.s1_entity_id = c.s1_entity_id AND tp.cand_entity_id = c.cand_entity_id
    """)
    con.unregister('batch_ids_view')
    gc.collect()

    if (i+1) % 10 == 0 or i == n_batches-1:
        print(f"  batch {i+1}/{n_batches}, {time.time()-t0:.0f}s elapsed")

print(f"build: {(time.time()-t0)/60:.1f} min")
n = con.execute("SELECT COUNT(*) FROM training_matrix").fetchone()[0]
n_pos = con.execute("SELECT COUNT(*) FROM training_matrix WHERE is_match").fetchone()[0]
print(f"rows: {n:,} (expected 411,043,867), positive: {n_pos:,} ({n_pos/n*100:.3f}%)")

In [ ]:
null_check = con.execute("""
SELECT
    SUM(CASE WHEN name_jaccard IS NULL THEN 1 ELSE 0 END) AS null_name_jac,
    SUM(CASE WHEN addr_jaccard IS NULL THEN 1 ELSE 0 END) AS null_addr_jac,
    SUM(CASE WHEN name_jw IS NULL THEN 1 ELSE 0 END) AS null_name_jw,
    SUM(CASE WHEN addr_jw IS NULL THEN 1 ELSE 0 END) AS null_addr_jw,
    SUM(CASE WHEN name_exact_approx IS NULL THEN 1 ELSE 0 END) AS null_exact_n,
    SUM(CASE WHEN addr_exact_approx IS NULL THEN 1 ELSE 0 END) AS null_exact_a
FROM training_matrix
""").fetchdf()
print(null_check)

In [ ]:
t0 = time.time()
con.execute(f"COPY training_matrix TO '{DATA_ROOT}/training_matrix.parquet' (FORMAT PARQUET)")
print(f"parquet export: {(time.time()-t0)/60:.1f} min")

import os
size_gb = os.path.getsize(f"{DATA_ROOT}/training_matrix.parquet") / 1e9
print(f"file size: {size_gb:.2f} GB")

In [ ]:
import pandas as pd, numpy as np, os, gc, time, shutil
import unicodedata, regex
import duckdb
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from rapidfuzz.distance import JaroWinkler

DATA_ROOT = "/home/ec2-user/SageMaker/data"
BASE = f"{DATA_ROOT}/dataset"

con = duckdb.connect('/home/ec2-user/SageMaker/blocking.duckdb')
con.execute("PRAGMA memory_limit='8GB'")
# Reconnecting to the same file in the same kernel reuses the open database, and DuckDB refuses to
# change its spill folder once it has spilled. So only set it if it isn't already this folder.
if con.execute("SELECT current_setting('temp_directory')").fetchone()[0] != '/home/ec2-user/SageMaker/tmp':
    con.execute("PRAGMA temp_directory='/home/ec2-user/SageMaker/tmp'")
con.execute("SET preserve_insertion_order=false")

print("Tables:", [t[0] for t in con.execute("SHOW TABLES").fetchall()])
n_tm = con.execute("SELECT COUNT(*) FROM training_matrix").fetchone()[0]
print(f"training_matrix rows: {n_tm:,} (expected 411,043,867 -- confirms disk survived)")
free_gb = shutil.disk_usage('/home/ec2-user/SageMaker').free / 1e9
print(f"free disk: {free_gb:.1f} GB")

In [ ]:
# Rebuild train/val entity split (lost from kernel memory, not from disk)
all_s1_ids = con.execute("SELECT DISTINCT entity_id FROM pairs_s1").fetchdf()['entity_id'].tolist()
train_ids, val_ids = train_test_split(all_s1_ids, test_size=0.15, random_state=42)
train_ids_df = pd.DataFrame({'entity_id': train_ids})
val_ids_df = pd.DataFrame({'entity_id': val_ids})
con.register('train_ids_view', train_ids_df)
con.register('val_ids_view', val_ids_df)
print(f"train entities: {len(train_ids):,}, val entities: {len(val_ids):,}")

In [ ]:
# Positives: same as before, cheap (only ~1.79% of rows, small result)
t0 = time.time()
con.execute("""
CREATE OR REPLACE TABLE train_positives AS
SELECT tm.* FROM training_matrix tm
JOIN train_ids_view t ON t.entity_id = tm.s1_entity_id
WHERE tm.is_match = true
""")
n_pos_train = con.execute("SELECT COUNT(*) FROM train_positives").fetchone()[0]
print(f"train positives: {n_pos_train:,}, {time.time()-t0:.1f}s")

In [ ]:
# Negatives: sample at ROW level but bound the scan with TABLESAMPLE (block-level, much cheaper
# than USING SAMPLE's full-table reservoir sampling), applied to the already train-entity-filtered CTE
t0 = time.time()
neg_target = n_pos_train * 15

con.execute("""
CREATE OR REPLACE TABLE train_negatives_pool AS
SELECT tm.* FROM training_matrix tm
JOIN train_ids_view t ON t.entity_id = tm.s1_entity_id
WHERE tm.is_match = false
""")
n_pool = con.execute("SELECT COUNT(*) FROM train_negatives_pool").fetchone()[0]
print(f"negative pool (train entities): {n_pool:,}, {time.time()-t0:.1f}s")

In [ ]:
t0 = time.time()
frac = min(1.0, neg_target / n_pool)
con.execute(f"""
CREATE OR REPLACE TABLE train_negatives AS
SELECT * FROM train_negatives_pool TABLESAMPLE {frac*100:.4f}% (bernoulli)
""")
n_neg_train = con.execute("SELECT COUNT(*) FROM train_negatives").fetchone()[0]
print(f"train negatives sampled: {n_neg_train:,} (target {neg_target:,}), {time.time()-t0:.1f}s")
con.execute("DROP TABLE train_negatives_pool")

In [ ]:
import pandas as pd, numpy as np, os, gc, time, shutil
import duckdb
import lightgbm as lgb
from sklearn.model_selection import train_test_split

DATA_ROOT = "/home/ec2-user/SageMaker/data"
con = duckdb.connect('/home/ec2-user/SageMaker/blocking.duckdb')
con.execute("PRAGMA memory_limit='6GB'")  # lower again -- 8GB left too little pandas headroom twice now
# Reconnecting to the same file in the same kernel reuses the open database, and DuckDB refuses to
# change its spill folder once it has spilled. So only set it if it isn't already this folder.
if con.execute("SELECT current_setting('temp_directory')").fetchone()[0] != '/home/ec2-user/SageMaker/tmp':
    con.execute("PRAGMA temp_directory='/home/ec2-user/SageMaker/tmp'")
con.execute("SET preserve_insertion_order=false")

print("Tables:", [t[0] for t in con.execute("SHOW TABLES").fetchall()])

In [ ]:
n_pos = con.execute("SELECT COUNT(*) FROM train_positives").fetchone()[0]
n_neg = con.execute("SELECT COUNT(*) FROM train_negatives").fetchone()[0]
print(f"train_positives: {n_pos:,}, train_negatives: {n_neg:,}")

t0 = time.time()
con.execute(f"COPY (SELECT * FROM train_positives UNION ALL SELECT * FROM train_negatives) TO '{DATA_ROOT}/train_matrix.parquet' (FORMAT PARQUET)")
print(f"train parquet export: {(time.time()-t0)/60:.1f} min")

In [ ]:
import pandas as pd, numpy as np, os, gc, time, shutil
import duckdb
import lightgbm as lgb

DATA_ROOT = "/home/ec2-user/SageMaker/data"
con = duckdb.connect('/home/ec2-user/SageMaker/blocking.duckdb')
con.execute("PRAGMA memory_limit='4GB'")
# Reconnecting to the same file in the same kernel reuses the open database, and DuckDB refuses to
# change its spill folder once it has spilled. So only set it if it isn't already this folder.
if con.execute("SELECT current_setting('temp_directory')").fetchone()[0] != '/home/ec2-user/SageMaker/tmp':
    con.execute("PRAGMA temp_directory='/home/ec2-user/SageMaker/tmp'")

print("Tables:", [t[0] for t in con.execute("SHOW TABLES").fetchall()])
free_gb = shutil.disk_usage('/home/ec2-user/SageMaker').free / 1e9
print(f"free disk: {free_gb:.1f} GB")

In [ ]:
n_pos = con.execute("SELECT COUNT(*) FROM train_positives").fetchone()[0]
print(f"train_positives: {n_pos:,}")

t0 = time.time()
con.execute(f"""
CREATE OR REPLACE TABLE train_negatives_5x AS
SELECT * FROM train_negatives LIMIT {n_pos*5}
""")
n_neg_5x = con.execute("SELECT COUNT(*) FROM train_negatives_5x").fetchone()[0]
print(f"train_negatives_5x: {n_neg_5x:,}, {time.time()-t0:.1f}s")

In [ ]:
t0 = time.time()
con.execute(f"COPY (SELECT * FROM train_positives UNION ALL SELECT * FROM train_negatives_5x) TO '{DATA_ROOT}/train_matrix_5x.parquet' (FORMAT PARQUET)")
print(f"5x parquet export: {(time.time()-t0)/60:.1f} min")

import os
size_gb = os.path.getsize(f"{DATA_ROOT}/train_matrix_5x.parquet") / 1e9
print(f"file size: {size_gb:.2f} GB")

In [ ]:
t0 = time.time()
train_df = con.execute(f"SELECT * FROM read_parquet('{DATA_ROOT}/train_matrix_5x.parquet')").fetchdf()
print(f"train_df loaded: {len(train_df):,} rows, {time.time()-t0:.1f}s")

feature_cols = ['name_jaccard','name_has_shared_rare_token','addr_jaccard','addr_has_shared_rare_token',
                 'name_exact_approx','addr_exact_approx','name_jw','addr_jw']
X_train = train_df[feature_cols].astype(float)
y_train = train_df['is_match'].astype(int)
print(f"X_train shape: {X_train.shape}, positive rate: {y_train.mean():.4f}")

del train_df
gc.collect()

In [ ]:
all_s1_ids = con.execute("SELECT DISTINCT entity_id FROM pairs_s1").fetchdf()['entity_id'].tolist()
from sklearn.model_selection import train_test_split
train_ids, val_ids = train_test_split(all_s1_ids, test_size=0.15, random_state=42)
val_ids_df = pd.DataFrame({'entity_id': val_ids})
con.register('val_ids_view', val_ids_df)
print(f"val entities: {len(val_ids):,}")

t0 = time.time()
con.execute(f"""
COPY (
    SELECT tm.* FROM training_matrix tm
    JOIN val_ids_view v ON v.entity_id = tm.s1_entity_id
) TO '{DATA_ROOT}/val_matrix.parquet' (FORMAT PARQUET)
""")
print(f"val parquet export: {(time.time()-t0)/60:.1f} min")

import os
size_gb = os.path.getsize(f"{DATA_ROOT}/val_matrix.parquet") / 1e9
print(f"val file size: {size_gb:.2f} GB")

In [ ]:
t0 = time.time()
val_df = con.execute(f"SELECT * FROM read_parquet('{DATA_ROOT}/val_matrix.parquet')").fetchdf()
print(f"val_df loaded: {len(val_df):,} rows, {time.time()-t0:.1f}s, positive rate: {val_df['is_match'].mean():.4f}")

X_val = val_df[feature_cols].astype(float)
y_val = val_df['is_match'].astype(int)
val_ids_ref = val_df[['s1_entity_id','cand_entity_id']].copy()
del val_df
gc.collect()

In [ ]:
train_set = lgb.Dataset(X_train, label=y_train, feature_name=feature_cols)
val_set = lgb.Dataset(X_val, label=y_val, feature_name=feature_cols, reference=train_set)

params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'is_unbalance': False,  # we already control balance via 5x undersampling, don't double-correct
    'verbose': -1,
    'num_threads': 4,  # cap threads -- avoid competing hard with host memory pressure
}

t0 = time.time()
model = lgb.train(
    params,
    train_set,
    num_boost_round=500,
    valid_sets=[train_set, val_set],
    valid_names=['train', 'val'],
    callbacks=[lgb.early_stopping(stopping_rounds=30), lgb.log_evaluation(period=25)]
)
print(f"training: {(time.time()-t0)/60:.1f} min, best iteration: {model.best_iteration}")

In [ ]:
y_val_prob = model.predict(X_val, num_iteration=model.best_iteration)
val_ids_ref['match_prob'] = y_val_prob
val_ids_ref['is_match'] = y_val.values
print(f"scored {len(val_ids_ref):,} pairs, prob range: [{y_val_prob.min():.4f}, {y_val_prob.max():.4f}]")

In [ ]:
import pandas as pd, numpy as np, os, gc, time, shutil
import duckdb
import lightgbm as lgb

DATA_ROOT = "/home/ec2-user/SageMaker/data"
con = duckdb.connect('/home/ec2-user/SageMaker/blocking.duckdb')
con.execute("PRAGMA memory_limit='4GB'")
# Reconnecting to the same file in the same kernel reuses the open database, and DuckDB refuses to
# change its spill folder once it has spilled. So only set it if it isn't already this folder.
if con.execute("SELECT current_setting('temp_directory')").fetchone()[0] != '/home/ec2-user/SageMaker/tmp':
    con.execute("PRAGMA temp_directory='/home/ec2-user/SageMaker/tmp'")

print("Tables:", [t[0] for t in con.execute("SHOW TABLES").fetchall()])
free_gb = shutil.disk_usage('/home/ec2-user/SageMaker').free / 1e9
print(f"free disk: {free_gb:.1f} GB")

In [ ]:
train_df = con.execute(f"SELECT * FROM read_parquet('{DATA_ROOT}/train_matrix_5x.parquet')").fetchdf()
feature_cols = ['name_jaccard','name_has_shared_rare_token','addr_jaccard','addr_has_shared_rare_token',
                 'name_exact_approx','addr_exact_approx','name_jw','addr_jw']
X_train = train_df[feature_cols].astype(float)
y_train = train_df['is_match'].astype(int)
del train_df
gc.collect()

val_df = con.execute(f"SELECT * FROM read_parquet('{DATA_ROOT}/val_matrix.parquet')").fetchdf()
X_val = val_df[feature_cols].astype(float)
y_val = val_df['is_match'].astype(int)
val_ids_ref = val_df[['s1_entity_id','cand_entity_id']].copy()
del val_df
gc.collect()
print(f"X_train: {X_train.shape}, X_val: {X_val.shape}")

In [ ]:
train_set = lgb.Dataset(X_train, label=y_train, feature_name=feature_cols)
val_set = lgb.Dataset(X_val, label=y_val, feature_name=feature_cols, reference=train_set)

params = {
    'objective': 'binary', 'metric': 'binary_logloss', 'boosting_type': 'gbdt',
    'num_leaves': 31, 'learning_rate': 0.05, 'feature_fraction': 0.9,
    'bagging_fraction': 0.8, 'bagging_freq': 5, 'verbose': -1, 'num_threads': 4,
}

t0 = time.time()
model = lgb.train(
    params, train_set, num_boost_round=500,
    valid_sets=[val_set], valid_names=['val'],
    callbacks=[lgb.early_stopping(stopping_rounds=30), lgb.log_evaluation(period=50)]
)
print(f"training: {(time.time()-t0)/60:.1f} min")

model.save_model(f"{DATA_ROOT}/lgb_model.txt")
print("model saved to disk")

In [ ]:
y_val_prob = model.predict(X_val, num_iteration=model.best_iteration)
val_ids_ref['match_prob'] = y_val_prob
val_ids_ref['is_match'] = y_val.values
print(f"scored {len(val_ids_ref):,} pairs, prob range: [{y_val_prob.min():.4f}, {y_val_prob.max():.4f}]")

con.register('val_scored_view', val_ids_ref)
con.execute("CREATE OR REPLACE TABLE val_scored AS SELECT * FROM val_scored_view")
con.unregister('val_scored_view')
print("val_scored checkpointed to blocking.duckdb")

In [ ]:
sorted_df = val_ids_ref.sort_values('match_prob', ascending=False).reset_index(drop=True)
s1_arr = sorted_df['s1_entity_id'].values
cand_arr = sorted_df['cand_entity_id'].values
prob_arr = sorted_df['match_prob'].values
is_match_arr = sorted_df['is_match'].values
del sorted_df
gc.collect()
print(f"prepared arrays: {len(s1_arr):,} rows")

In [ ]:
def per_entity_f05_arrays(s1_arr, cand_arr, prob_arr, is_match_arr, threshold):
    above_thresh = prob_arr >= threshold
    used_s1 = set()
    used_cand = set()
    pred_arr = np.zeros(len(s1_arr), dtype=bool)
    for i in range(len(s1_arr)):
        if not above_thresh[i]:
            continue
        s1, cand = s1_arr[i], cand_arr[i]
        if s1 in used_s1 or cand in used_cand:
            continue
        used_s1.add(s1)
        used_cand.add(cand)
        pred_arr[i] = True

    agg_df = pd.DataFrame({'s1_entity_id': s1_arr, 'true_match': is_match_arr, 'predicted': pred_arr})
    agg = agg_df.groupby('s1_entity_id').agg(
        true_pos=('true_match', 'sum'),
        pred_pos=('predicted', 'sum'),
    )
    tp_series = agg_df[agg_df['predicted'] & (agg_df['true_match'] == 1)].groupby('s1_entity_id').size()
    agg['tp'] = tp_series.reindex(agg.index, fill_value=0)
    del agg_df
    gc.collect()

    true_pos = agg['true_pos'].values
    pred_pos = agg['pred_pos'].values
    tp = agg['tp'].values

    f05 = np.zeros(len(agg))
    f05[(true_pos == 0) & (pred_pos == 0)] = 1.0
    mask = (true_pos > 0) & (pred_pos > 0)
    precision = np.divide(tp[mask], pred_pos[mask])
    recall = np.divide(tp[mask], true_pos[mask])
    beta = 0.5
    denom = beta**2 * precision + recall
    f05_vals = np.where(denom > 0, (1 + beta**2) * precision * recall / np.where(denom == 0, 1, denom), 0.0)
    f05[mask] = f05_vals
    return f05.mean(), len(agg)

t0 = time.time()
for thresh in [0.3, 0.5, 0.7, 0.9]:
    score, n_entities = per_entity_f05_arrays(s1_arr, cand_arr, prob_arr, is_match_arr, thresh)
    print(f"threshold={thresh}: macro F0.5={score:.4f} (n={n_entities:,} entities), {time.time()-t0:.0f}s elapsed")

In [ ]:
all_s1_ids = con.execute("SELECT DISTINCT entity_id FROM pairs_s1").fetchdf()['entity_id'].tolist()
from sklearn.model_selection import train_test_split
train_ids, val_ids = train_test_split(all_s1_ids, test_size=0.15, random_state=42)

n_val_total = len(val_ids)
n_val_scored = len(np.unique(s1_arr))
print(f"val entities total: {n_val_total:,}")
print(f"entities present in val_ids_ref (have >=1 candidate): {n_val_scored:,}")
print(f"missing entities (zero candidates in training_matrix): {n_val_total - n_val_scored:,}")

In [ ]:
# Among val entities WITH candidates, how many have zero TRUE positive candidates
# (i.e. the true match was dropped by blocking) vs at least one true candidate present?
agg_check = pd.DataFrame({'s1_entity_id': s1_arr, 'true_match': is_match_arr}).groupby('s1_entity_id')['true_match'].sum()
n_has_true_candidate = (agg_check > 0).sum()
n_no_true_candidate = (agg_check == 0).sum()
print(f"val entities WITH true match reachable (>=1 true candidate present): {n_has_true_candidate:,}")
print(f"val entities with NO true candidate present (blocking missed it, or true singleton): {n_no_true_candidate:,}")

In [ ]:
BASE = f"{DATA_ROOT}/dataset"
gt = pd.read_csv(f"{BASE}/train/train_ground_truth.tsv", sep="\t")
gt['matched_entity_ids'] = gt['matched_entity_ids'].fillna('')
gt['match_list'] = gt['matched_entity_ids'].apply(lambda x: [i.strip() for i in x.split(',') if i.strip() != ''])
print(f"gt loaded: {len(gt):,} rows")

In [ ]:
# Distinguish true singletons (no ground-truth match at all) from blocking misses
# (has a ground-truth match, but it's not among this entity's candidates)
val_gt = gt[gt['source1_entity_id'].astype(str).isin(val_ids)][['source1_entity_id', 'match_list']].copy()
val_gt['source1_entity_id'] = val_gt['source1_entity_id'].astype(str)
val_gt['has_gt_match'] = val_gt['match_list'].apply(lambda x: len(x) > 0)

no_true_candidate_ids = agg_check[agg_check == 0].index
val_gt_lookup = val_gt.set_index('source1_entity_id')['has_gt_match']

is_true_singleton = val_gt_lookup.reindex(no_true_candidate_ids).fillna(False)
n_true_singletons = is_true_singleton.sum()
n_blocking_misses = (~is_true_singleton).sum()

print(f"of the 20,082 with no true candidate present:")
print(f"  true singletons (no gt match at all -- correctly unreachable, fine): {n_true_singletons:,}")
print(f"  blocking misses (gt match exists, but not in candidates -- real ceiling): {n_blocking_misses:,}")
print(f"blocking-miss rate among val entities: {n_blocking_misses/len(s1_arr.__class__(np.unique(s1_arr)))*100:.2f}%" if False else f"blocking-miss rate: {n_blocking_misses/331023*100:.2f}%")

In [ ]:
def per_entity_top1_f05(s1_arr, cand_arr, prob_arr, is_match_arr, min_prob=0.0):
    df = pd.DataFrame({'s1_entity_id': s1_arr, 'cand_entity_id': cand_arr, 'prob': prob_arr, 'true_match': is_match_arr})
    idx = df.groupby('s1_entity_id')['prob'].idxmax()
    top1 = df.loc[idx].copy()
    top1['predicted'] = top1['prob'] >= min_prob

    agg = df.groupby('s1_entity_id')['true_match'].sum().rename('true_pos')
    top1 = top1.set_index('s1_entity_id')
    top1['true_pos'] = agg.reindex(top1.index)
    top1['pred_pos'] = top1['predicted'].astype(int)
    top1['tp'] = ((top1['predicted']) & (top1['true_match'] == 1)).astype(int)

    true_pos = top1['true_pos'].values
    pred_pos = top1['pred_pos'].values
    tp = top1['tp'].values

    f05 = np.zeros(len(top1))
    f05[(true_pos == 0) & (pred_pos == 0)] = 1.0
    mask = (true_pos > 0) & (pred_pos > 0)
    precision = np.divide(tp[mask], pred_pos[mask])
    recall = np.divide(tp[mask], true_pos[mask])
    beta = 0.5
    denom = beta**2 * precision + recall
    f05_vals = np.where(denom > 0, (1 + beta**2) * precision * recall / np.where(denom == 0, 1, denom), 0.0)
    f05[mask] = f05_vals
    return f05.mean()

t0 = time.time()
for min_prob in [0.0, 0.3, 0.5, 0.7, 0.9]:
    score = per_entity_top1_f05(s1_arr, cand_arr, prob_arr, is_match_arr, min_prob)
    print(f"top1, min_prob={min_prob}: macro F0.5={score:.4f}, {time.time()-t0:.0f}s elapsed")

In [ ]:
print("pair_index:")
print(con.execute("SELECT * FROM pair_index LIMIT 3").fetchdf())
print()
print("pairs_s1:")
print(con.execute("SELECT * FROM pairs_s1 LIMIT 3").fetchdf())
print()
print("pair_postings:")
print(con.execute("SELECT * FROM pair_postings LIMIT 3").fetchdf())
print()
print("candidates_final columns:")
print(con.execute("DESCRIBE candidates_final").fetchdf())

In [ ]:
print([t for t in ['all_true_pairs','s1_sample'] if t not in [x[0] for x in con.execute("SHOW TABLES").fetchall()]], "missing (empty list = both present)")

In [ ]:
def top_k_recall(top_k, con):
    t0 = time.time()
    con.execute(f"""
    CREATE OR REPLACE TEMP TABLE ranked_test AS
    WITH joined AS (
        SELECT ps.entity_id AS s1_entity_id, pp.entity_id AS cand_entity_id,
               MIN(pi.doc_freq) AS best_freq
        FROM pairs_s1 ps
        JOIN s1_sample ss ON ss.entity_id = ps.entity_id
        JOIN pair_index pi ON pi.field = ps.field AND pi.token_a = ps.token_a AND pi.token_b = ps.token_b
        JOIN pair_postings pp ON pp.field = pi.field AND pp.token_a = pi.token_a AND pp.token_b = pi.token_b
        GROUP BY ps.entity_id, pp.entity_id
    ),
    ranked AS (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY s1_entity_id ORDER BY best_freq ASC, cand_entity_id ASC) AS rnk
        FROM joined
    )
    SELECT s1_entity_id, cand_entity_id FROM ranked WHERE rnk <= {top_k}
    """)

    n_full_recall = con.execute("""
    WITH true_per_entity AS (
        SELECT tp.s1_entity_id, COUNT(*) AS n_true
        FROM all_true_pairs tp
        JOIN s1_sample ss ON ss.entity_id = tp.s1_entity_id
        GROUP BY tp.s1_entity_id
    ),
    caught_per_entity AS (
        SELECT tp.s1_entity_id, COUNT(*) AS n_caught
        FROM all_true_pairs tp
        JOIN s1_sample ss ON ss.entity_id = tp.s1_entity_id
        JOIN ranked_test rt ON rt.s1_entity_id = tp.s1_entity_id AND rt.cand_entity_id = tp.cand_entity_id
        GROUP BY tp.s1_entity_id
    )
    SELECT COUNT(*) FROM true_per_entity t
    LEFT JOIN caught_per_entity c ON c.s1_entity_id = t.s1_entity_id
    WHERE t.n_true = COALESCE(c.n_caught, 0)
    """).fetchone()[0]

    n_mean_cand = con.execute("SELECT COUNT(*) * 1.0 / 5000 FROM ranked_test").fetchone()[0]
    con.execute("DROP TABLE ranked_test")
    return n_full_recall, n_mean_cand, time.time() - t0

for top_k in [200, 300, 500]:
    n_recall, mean_cand, elapsed = top_k_recall(top_k, con)
    print(f"TOP_K={top_k}: full-recall={n_recall}/5000 ({n_recall/5000*100:.2f}%), mean_candidates={mean_cand:.1f}, {elapsed:.1f}s")

In [ ]:
# Direct comparison: does candidates_final itself (the real, validated table) give 89.94% on this same 5K sample?
n_real_recall = con.execute("""
WITH true_per_entity AS (
    SELECT tp.s1_entity_id, COUNT(*) AS n_true
    FROM all_true_pairs tp
    JOIN s1_sample ss ON ss.entity_id = tp.s1_entity_id
    GROUP BY tp.s1_entity_id
),
caught_per_entity AS (
    SELECT tp.s1_entity_id, COUNT(*) AS n_caught
    FROM all_true_pairs tp
    JOIN s1_sample ss ON ss.entity_id = tp.s1_entity_id
    JOIN candidates_final cf ON cf.s1_entity_id = tp.s1_entity_id AND cf.cand_entity_id = tp.cand_entity_id
    GROUP BY tp.s1_entity_id
)
SELECT COUNT(*) FROM true_per_entity t
LEFT JOIN caught_per_entity c ON c.s1_entity_id = t.s1_entity_id
WHERE t.n_true = COALESCE(c.n_caught, 0)
""").fetchone()[0]
print(f"candidates_final full-recall on 5K sample: {n_real_recall}/5000 ({n_real_recall/5000*100:.2f}%)")

In [ ]:
BASE = f"{DATA_ROOT}/dataset"
if 'gt' not in dir():
    gt = pd.read_csv(f"{BASE}/train/train_ground_truth.tsv", sep="\t")
    gt['matched_entity_ids'] = gt['matched_entity_ids'].fillna('')
    gt['match_list'] = gt['matched_entity_ids'].apply(lambda x: [i.strip() for i in x.split(',') if i.strip() != ''])

fresh_sample = gt.sample(5000, random_state=2)['source1_entity_id'].astype(str).tolist()
current_s1_sample = con.execute("SELECT entity_id FROM s1_sample").fetchdf()['entity_id'].tolist()

print(f"fresh gt.sample(5000, random_state=2) size: {len(fresh_sample)}")
print(f"current s1_sample size: {len(current_s1_sample)}")
print(f"overlap: {len(set(fresh_sample) & set(current_s1_sample))}")
print(f"identical sets: {set(fresh_sample) == set(current_s1_sample)}")

In [ ]:
# Rebuild s1_sample with the CORRECT method matching the original 89.94% validation
correct_sample_ids = gt.sample(5000, random_state=2)['source1_entity_id'].astype(str).tolist()
correct_sample_df = pd.DataFrame({'entity_id': correct_sample_ids})
con.execute("CREATE OR REPLACE TABLE s1_sample AS SELECT * FROM correct_sample_df")
n_check = con.execute("SELECT COUNT(*) FROM s1_sample").fetchone()[0]
print(f"s1_sample rebuilt: {n_check} rows")

# Re-verify baseline first
n_baseline = con.execute("""
WITH true_per_entity AS (
    SELECT tp.s1_entity_id, COUNT(*) AS n_true
    FROM all_true_pairs tp
    JOIN s1_sample ss ON ss.entity_id = tp.s1_entity_id
    GROUP BY tp.s1_entity_id
),
caught_per_entity AS (
    SELECT tp.s1_entity_id, COUNT(*) AS n_caught
    FROM all_true_pairs tp
    JOIN s1_sample ss ON ss.entity_id = tp.s1_entity_id
    JOIN candidates_final cf ON cf.s1_entity_id = tp.s1_entity_id AND cf.cand_entity_id = tp.cand_entity_id
    GROUP BY tp.s1_entity_id
)
SELECT COUNT(*) FROM true_per_entity t
LEFT JOIN caught_per_entity c ON c.s1_entity_id = t.s1_entity_id
WHERE t.n_true = COALESCE(c.n_caught, 0)
""").fetchone()[0]
print(f"candidates_final full-recall on CORRECTED 5K sample: {n_baseline}/5000 ({n_baseline/5000*100:.2f}%) -- should be ~89.94%")

In [ ]:
n_true_pairs = con.execute("SELECT COUNT(*) FROM all_true_pairs").fetchone()[0]
print(f"all_true_pairs rows: {n_true_pairs:,} (expected 7,638,365)")

# Also double check the true pair count specifically for this sample
n_true_for_sample = con.execute("""
SELECT COUNT(*) FROM all_true_pairs tp
JOIN s1_sample ss ON ss.entity_id = tp.s1_entity_id
""").fetchone()[0]
print(f"true pairs for this 5K sample: {n_true_for_sample:,} (earlier tonight's number was 17,273 -- but that was the OLD/wrong sample)")

In [ ]:
for top_k in [200, 300, 500, 1000]:
    n_recall, mean_cand, elapsed = top_k_recall(top_k, con)
    print(f"TOP_K={top_k}: full-recall={n_recall}/5000 ({n_recall/5000*100:.2f}%), mean_candidates={mean_cand:.1f}, {elapsed:.1f}s")

In [ ]:
print([t for t in ['tokens','tokens_s1','token_freq','tokens_top','tokens_top_s1'] if t not in [x[0] for x in con.execute("SHOW TABLES").fetchall()]], "missing (empty = all present)")

In [ ]:
print(con.execute("SELECT COUNT(*) FROM tokens").fetchone()[0], "tokens rows (S2+S3, all tokens)")
print(con.execute("SELECT COUNT(*) FROM tokens_s1").fetchone()[0], "tokens_s1 rows (S1, all tokens)")
print(con.execute("SELECT entity_id, field, token FROM tokens LIMIT 5").fetchdf())

In [ ]:
# Total corpus size for IDF (N = total documents per field, for idf = log(N / doc_freq))
n_cand_docs = con.execute("SELECT COUNT(DISTINCT entity_id) FROM tokens").fetchone()[0]
print(f"N candidate docs (S2+S3): {n_cand_docs:,}")

In [ ]:
free_gb = shutil.disk_usage('/home/ec2-user/SageMaker').free / 1e9
print(f"free disk: {free_gb:.1f} GB")

import psutil
mem = psutil.virtual_memory()
print(f"available RAM: {mem.available/1e9:.1f} GB / {mem.total/1e9:.1f} GB total ({mem.percent}% used)")

In [ ]:
con.execute("PRAGMA memory_limit='3GB'")  # leave ~2GB headroom given 5.2GB available

In [ ]:
t0 = time.time()
con.execute("""
CREATE OR REPLACE TABLE tfidf_cand AS
SELECT t.entity_id, t.field, t.token,
       COUNT(*) AS tf,
       LN(10320219.0 / f.doc_freq) AS idf
FROM tokens t
JOIN token_freq f ON f.token = t.token
GROUP BY t.entity_id, t.field, t.token, f.doc_freq
""")
print(f"tfidf_cand: {(time.time()-t0)/60:.1f} min, rows: {con.execute('SELECT COUNT(*) FROM tfidf_cand').fetchone()[0]:,}")

In [ ]:
t0 = time.time()
con.execute("""
CREATE OR REPLACE TABLE token_freq_s1 AS
SELECT token, COUNT(DISTINCT entity_id) AS doc_freq FROM tokens_s1 GROUP BY token
""")
n_s1_docs = con.execute("SELECT COUNT(DISTINCT entity_id) FROM tokens_s1").fetchone()[0]
print(f"token_freq_s1: {(time.time()-t0)/60:.1f} min, N_s1_docs={n_s1_docs:,}")

t0 = time.time()
con.execute(f"""
CREATE OR REPLACE TABLE tfidf_s1 AS
SELECT t.entity_id, t.field, t.token,
       COUNT(*) AS tf,
       LN({n_s1_docs}.0 / f.doc_freq) AS idf
FROM tokens_s1 t
JOIN token_freq_s1 f ON f.token = t.token
GROUP BY t.entity_id, t.field, t.token, f.doc_freq
""")
print(f"tfidf_s1: {(time.time()-t0)/60:.1f} min, rows: {con.execute('SELECT COUNT(*) FROM tfidf_s1').fetchone()[0]:,}")

In [ ]:
t0 = time.time()
con.execute("""
CREATE OR REPLACE TABLE tfidf_norms_cand AS
SELECT entity_id, field, SQRT(SUM((tf*idf)*(tf*idf))) AS norm
FROM tfidf_cand GROUP BY entity_id, field
""")
con.execute("""
CREATE OR REPLACE TABLE tfidf_norms_s1 AS
SELECT entity_id, field, SQRT(SUM((tf*idf)*(tf*idf))) AS norm
FROM tfidf_s1 GROUP BY entity_id, field
""")
print(f"norms built: {(time.time()-t0)/60:.1f} min")
print("norms_cand rows:", con.execute("SELECT COUNT(*) FROM tfidf_norms_cand").fetchone()[0])
print("norms_s1 rows:", con.execute("SELECT COUNT(*) FROM tfidf_norms_s1").fetchone()[0])

In [ ]:
con.execute("DROP TABLE IF EXISTS tfidf_cosine")
con.execute("""CREATE OR REPLACE TABLE tfidf_cosine (
    s1_entity_id VARCHAR, cand_entity_id VARCHAR,
    name_tfidf_cosine DOUBLE, addr_tfidf_cosine DOUBLE
)""")

all_ids = con.execute("SELECT DISTINCT entity_id FROM pairs_s1").fetchdf()['entity_id'].tolist()
BATCH_SIZE = 20_000
n_batches = int(np.ceil(len(all_ids) / BATCH_SIZE))
print(f"{len(all_ids):,} entities, {n_batches} batches")

t0 = time.time()
for i in range(n_batches):
    batch_ids = all_ids[i*BATCH_SIZE:(i+1)*BATCH_SIZE]
    batch_ids_df = pd.DataFrame({'entity_id': batch_ids})
    con.register('batch_ids_view', batch_ids_df)

    con.execute("""
    INSERT INTO tfidf_cosine
    WITH batch_pairs AS (
        SELECT c.s1_entity_id, c.cand_entity_id
        FROM candidates_final c
        JOIN batch_ids_view b ON b.entity_id = c.s1_entity_id
    ),
    batch_tfidf_s1 AS (SELECT t.* FROM tfidf_s1 t JOIN batch_ids_view b ON b.entity_id = t.entity_id),
    batch_tfidf_cand AS (
        SELECT t.* FROM tfidf_cand t
        JOIN (SELECT DISTINCT cand_entity_id FROM batch_pairs) bc ON bc.cand_entity_id = t.entity_id
    ),
    dot_products AS (
        SELECT bp.s1_entity_id, bp.cand_entity_id, s.field,
               SUM(s.tf * s.idf * c.tf * c.idf) AS dot
        FROM batch_pairs bp
        JOIN batch_tfidf_s1 s ON s.entity_id = bp.s1_entity_id
        JOIN batch_tfidf_cand c ON c.entity_id = bp.cand_entity_id AND c.field = s.field AND c.token = s.token
        GROUP BY bp.s1_entity_id, bp.cand_entity_id, s.field
    )
    SELECT
        bp.s1_entity_id, bp.cand_entity_id,
        COALESCE(dn.dot / NULLIF(ns1n.norm * ncn.norm, 0), 0) AS name_tfidf_cosine,
        COALESCE(da.dot / NULLIF(ns1a.norm * nca.norm, 0), 0) AS addr_tfidf_cosine
    FROM batch_pairs bp
    LEFT JOIN dot_products dn ON dn.s1_entity_id = bp.s1_entity_id AND dn.cand_entity_id = bp.cand_entity_id AND dn.field = 'N'
    LEFT JOIN dot_products da ON da.s1_entity_id = bp.s1_entity_id AND da.cand_entity_id = bp.cand_entity_id AND da.field = 'A'
    LEFT JOIN tfidf_norms_s1 ns1n ON ns1n.entity_id = bp.s1_entity_id AND ns1n.field = 'N'
    LEFT JOIN tfidf_norms_cand ncn ON ncn.entity_id = bp.cand_entity_id AND ncn.field = 'N'
    LEFT JOIN tfidf_norms_s1 ns1a ON ns1a.entity_id = bp.s1_entity_id AND ns1a.field = 'A'
    LEFT JOIN tfidf_norms_cand nca ON nca.entity_id = bp.cand_entity_id AND nca.field = 'A'
    """)
    con.unregister('batch_ids_view')
    gc.collect()

    if (i+1) % 10 == 0 or i == n_batches-1:
        print(f"  batch {i+1}/{n_batches}, {time.time()-t0:.0f}s elapsed")

print(f"build: {(time.time()-t0)/60:.1f} min")
n = con.execute("SELECT COUNT(*) FROM tfidf_cosine").fetchone()[0]
print(f"rows: {n:,} (expected 411,043,867)")

In [ ]:
print([t[0] for t in con.execute("SHOW TABLES").fetchall()])

In [ ]:
t0 = time.time()
con.execute(f"""
CREATE OR REPLACE TABLE train_matrix_with_tfidf AS
SELECT tm.*, tc.name_tfidf_cosine, tc.addr_tfidf_cosine
FROM read_parquet('{DATA_ROOT}/train_matrix_5x.parquet') tm
LEFT JOIN tfidf_cosine tc
  ON tc.s1_entity_id = tm.s1_entity_id AND tc.cand_entity_id = tm.cand_entity_id
""")
n = con.execute("SELECT COUNT(*) FROM train_matrix_with_tfidf").fetchone()[0]
n_null = con.execute("SELECT COUNT(*) FROM train_matrix_with_tfidf WHERE name_tfidf_cosine IS NULL").fetchone()[0]
print(f"train_matrix_with_tfidf: {(time.time()-t0)/60:.1f} min, rows: {n:,}, null cosine: {n_null:,}")

In [ ]:
t0 = time.time()
con.execute(f"""
CREATE OR REPLACE TABLE val_matrix_with_tfidf AS
SELECT vm.*, tc.name_tfidf_cosine, tc.addr_tfidf_cosine
FROM read_parquet('{DATA_ROOT}/val_matrix.parquet') vm
LEFT JOIN tfidf_cosine tc
  ON tc.s1_entity_id = vm.s1_entity_id AND tc.cand_entity_id = vm.cand_entity_id
""")
n = con.execute("SELECT COUNT(*) FROM val_matrix_with_tfidf").fetchone()[0]
n_null = con.execute("SELECT COUNT(*) FROM val_matrix_with_tfidf WHERE name_tfidf_cosine IS NULL").fetchone()[0]
print(f"val_matrix_with_tfidf: {(time.time()-t0)/60:.1f} min, rows: {n:,}, null cosine: {n_null:,}")

In [ ]:
t0 = time.time()
con.execute(f"COPY train_matrix_with_tfidf TO '{DATA_ROOT}/train_matrix_5x_tfidf.parquet' (FORMAT PARQUET)")
con.execute(f"COPY val_matrix_with_tfidf TO '{DATA_ROOT}/val_matrix_tfidf.parquet' (FORMAT PARQUET)")
print(f"exported: {(time.time()-t0)/60:.1f} min")

In [ ]:
train_df = con.execute(f"SELECT * FROM read_parquet('{DATA_ROOT}/train_matrix_5x_tfidf.parquet')").fetchdf()
feature_cols = ['name_jaccard','name_has_shared_rare_token','addr_jaccard','addr_has_shared_rare_token',
                 'name_exact_approx','addr_exact_approx','name_jw','addr_jw',
                 'name_tfidf_cosine','addr_tfidf_cosine']
X_train = train_df[feature_cols].astype(float)
y_train = train_df['is_match'].astype(int)
del train_df
gc.collect()

val_df = con.execute(f"SELECT * FROM read_parquet('{DATA_ROOT}/val_matrix_tfidf.parquet')").fetchdf()
X_val = val_df[feature_cols].astype(float)
y_val = val_df['is_match'].astype(int)
val_ids_ref = val_df[['s1_entity_id','cand_entity_id']].copy()
del val_df
gc.collect()
print(f"X_train: {X_train.shape}, X_val: {X_val.shape}")

In [ ]:
import pandas as pd, numpy as np, os, gc, time, shutil
import duckdb
DATA_ROOT = "/home/ec2-user/SageMaker/data"
con = duckdb.connect('/home/ec2-user/SageMaker/blocking.duckdb')
con.execute("PRAGMA memory_limit='3GB'")

import psutil
mem = psutil.virtual_memory()
print(f"available RAM: {mem.available/1e9:.2f} GB / {mem.total/1e9:.2f} GB ({mem.percent}% used)")

In [ ]:
# Confirm the exported parquets survived the restart (they're on disk, should be untouched)
print([t[0] for t in con.execute("SHOW TABLES").fetchall() if 'tfidf' in t[0]])
print(os.path.exists(f"{DATA_ROOT}/train_matrix_5x_tfidf.parquet"), os.path.exists(f"{DATA_ROOT}/val_matrix_tfidf.parquet"))

In [ ]:
t0 = time.time()
train_arrow = con.execute(f"SELECT {cols_sql}, is_match FROM read_parquet('{DATA_ROOT}/train_matrix_5x_tfidf.parquet')").arrow().read_all()
X_train = train_arrow.select(feature_cols).to_pandas()
y_train = train_arrow.column('is_match').to_pandas().astype(int)
del train_arrow
gc.collect()
print(f"train loaded: {(time.time()-t0)/60:.1f} min, X_train: {X_train.shape}")

In [ ]:
t0 = time.time()
val_arrow = con.execute(f"SELECT {cols_sql}, is_match, s1_entity_id, cand_entity_id FROM read_parquet('{DATA_ROOT}/val_matrix_tfidf.parquet')").arrow().read_all()
X_val = val_arrow.select(feature_cols).to_pandas()
val_ids_ref = val_arrow.select(['s1_entity_id','cand_entity_id']).to_pandas()
y_val = val_arrow.column('is_match').to_pandas().astype(int)
del val_arrow
gc.collect()
print(f"val loaded: {(time.time()-t0)/60:.1f} min, X_val: {X_val.shape}")

In [ ]:
import lightgbm as lgb

t0 = time.time()
lgb_train = lgb.Dataset(X_train, label=y_train)
params = {
    'objective': 'binary',
    'metric': 'binary_logloss',
    'num_leaves': 63,
    'learning_rate': 0.05,
    'verbose': -1,
}
model = lgb.train(params, lgb_train, num_boost_round=200)
print(f"trained: {(time.time()-t0)/60:.1f} min")

In [ ]:
t0 = time.time()
prob_val = model.predict(X_val)
print(f"predicted: {(time.time()-t0)/60:.1f} min")

s1_arr = val_ids_ref['s1_entity_id'].values
cand_arr = val_ids_ref['cand_entity_id'].values
is_match_arr = y_val.values
prob_arr = prob_val

In [ ]:
def macro_f05(s1_arr, prob_arr, is_match_arr, threshold):
    df = pd.DataFrame({'s1_entity_id': s1_arr, 'prob': prob_arr, 'true_match': is_match_arr})
    df['predicted'] = df['prob'] >= threshold

    agg = df.groupby('s1_entity_id').agg(
        true_pos=('true_match', 'sum'),
        pred_pos=('predicted', 'sum'),
        tp=('predicted', lambda x: None)  # placeholder, computed below
    )
    tp_series = df[df['predicted'] & (df['true_match'] == 1)].groupby('s1_entity_id').size()
    agg['tp'] = tp_series.reindex(agg.index).fillna(0)

    true_pos = agg['true_pos'].values
    pred_pos = agg['pred_pos'].values
    tp = agg['tp'].values

    f05 = np.zeros(len(agg))
    f05[(true_pos == 0) & (pred_pos == 0)] = 1.0
    mask = (true_pos > 0) & (pred_pos > 0)
    precision = np.divide(tp[mask], pred_pos[mask])
    recall = np.divide(tp[mask], true_pos[mask])
    beta = 0.5
    denom = beta**2 * precision + recall
    f05_vals = np.where(denom > 0, (1 + beta**2) * precision * recall / np.where(denom == 0, 1, denom), 0.0)
    f05[mask] = f05_vals
    return f05.mean()

t0 = time.time()
for threshold in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    score = macro_f05(s1_arr, prob_arr, is_match_arr, threshold)
    print(f"threshold={threshold}: macro F0.5={score:.4f}, {time.time()-t0:.0f}s elapsed")

In [ ]:
t0 = time.time()
for threshold in [0.8, 0.85, 0.9, 0.92, 0.95, 0.97, 0.99]:
    score = macro_f05(s1_arr, prob_arr, is_match_arr, threshold)
    print(f"threshold={threshold}: macro F0.5={score:.4f}, {time.time()-t0:.0f}s elapsed")

In [ ]:
for threshold in [0.8, 0.9, 0.95, 0.99]:
    n_pred = (prob_arr >= threshold).sum()
    print(f"threshold={threshold}: {n_pred:,} predicted positive out of {len(prob_arr):,} candidate rows")

In [ ]:
t0 = time.time()
for threshold in [0.93, 0.94, 0.95, 0.96]:
    score = macro_f05(s1_arr, prob_arr, is_match_arr, threshold)
    n_pred = (prob_arr >= threshold).sum()
    print(f"threshold={threshold}: macro F0.5={score:.4f}, n_pred={n_pred:,}, {time.time()-t0:.0f}s elapsed")

In [ ]:
import os
test_dir = f"{DATA_ROOT}/dataset/test"
print(os.path.exists(test_dir))
if os.path.exists(test_dir):
    print(os.listdir(test_dir))

In [ ]:
t0 = time.time()
test_s1 = pd.read_csv(f"{DATA_ROOT}/dataset/test/test_source1.tsv", sep="\t")
test_s2 = pd.read_csv(f"{DATA_ROOT}/dataset/test/test_source2.tsv", sep="\t")
test_s3 = pd.read_csv(f"{DATA_ROOT}/dataset/test/test_source3.tsv", sep="\t")
print(f"loaded: {(time.time()-t0):.0f}s")
print(f"test_s1: {len(test_s1):,} rows, columns: {list(test_s1.columns)}")
print(f"test_s2: {len(test_s2):,} rows")
print(f"test_s3: {len(test_s3):,} rows")
print(f"test_s1 countries: {test_s1['country'].value_counts().to_dict()}")
print(f"test_s2 countries: {test_s2['country'].value_counts().to_dict()}")
print(f"test_s3 countries: {test_s3['country'].value_counts().to_dict()}")

In [ ]:
import psutil
print(f"SageMaker available RAM: {psutil.virtual_memory().available/1e9:.2f} GB")

In [ ]:
import unicodedata
import regex

TOKEN_RE = regex.compile(r'[\p{L}\p{M}\p{N}]+')

def tokenize(s):
    if pd.isnull(s):
        return []
    s = unicodedata.normalize('NFC', str(s)).lower()
    return TOKEN_RE.findall(s)

In [ ]:
# Step 1: tokenize test S2+S3 (candidate side) -- same tokenizer, same chunking
CHUNK_SIZE = 300_000

def load_tokens_chunked(df, field_name, col, table_name, chunk_size=CHUNK_SIZE):
    n_chunks = int(np.ceil(len(df) / chunk_size))
    for i in range(n_chunks):
        chunk = df.iloc[i*chunk_size : (i+1)*chunk_size]
        tmp = pd.DataFrame({
            'entity_id': chunk['entity_id'].astype(str).values,
            'token': chunk[col].apply(tokenize).values
        })
        tmp = tmp.explode('token').dropna(subset=['token'])
        tmp['field'] = field_name
        con.append(table_name, tmp[['entity_id', 'field', 'token']])
        del chunk, tmp
        gc.collect()
        if i % 5 == 0:
            print(f"  {field_name}/{col}: chunk {i+1}/{n_chunks} done")

con.execute("CREATE OR REPLACE TABLE tokens_test (entity_id VARCHAR, field VARCHAR, token VARCHAR)")

t0 = time.time()
for df in [test_s2, test_s3]:
    load_tokens_chunked(df, 'N', 'business_name', 'tokens_test')
    load_tokens_chunked(df, 'A', 'business_address', 'tokens_test')
print(f"tokens_test load took {(time.time()-t0)/60:.1f} min")
print(con.execute("SELECT COUNT(*) FROM tokens_test").fetchone())

In [ ]:
MAX_TOKENS = 5

con.execute("""
CREATE OR REPLACE TABLE token_freq_test AS
SELECT token, COUNT(DISTINCT entity_id) AS doc_freq FROM tokens_test GROUP BY token
""")

con.execute(f"""
CREATE OR REPLACE TABLE tokens_top_test AS
SELECT entity_id, field, token, doc_freq FROM (
    SELECT t.entity_id, t.field, t.token, f.doc_freq,
           ROW_NUMBER() OVER (PARTITION BY t.entity_id, t.field ORDER BY f.doc_freq ASC, t.token) AS rnk
    FROM tokens_test t JOIN token_freq_test f USING (token)
) WHERE rnk <= {MAX_TOKENS}
""")
print("tokens_top_test rows:", con.execute("SELECT COUNT(*) FROM tokens_top_test").fetchone())

In [ ]:
t0 = time.time()
con.execute("""
CREATE OR REPLACE TABLE pair_freq_test AS
SELECT a.field, a.token AS token_a, b.token AS token_b,
       COUNT(DISTINCT a.entity_id) AS doc_freq
FROM tokens_top_test a JOIN tokens_top_test b
  ON a.entity_id = b.entity_id AND a.field = b.field AND a.token < b.token
GROUP BY a.field, a.token, b.token
""")
print(f"pair_freq_test build took {(time.time()-t0)/60:.1f} min")
print("distinct pairs:", con.execute("SELECT COUNT(*) FROM pair_freq_test").fetchone())

In [ ]:
PAIR_CAP = 2000
con.execute(f"""
CREATE OR REPLACE TABLE pair_index_test AS
SELECT field, token_a, token_b, doc_freq
FROM pair_freq_test
WHERE doc_freq <= {PAIR_CAP}
""")
print("pair_index_test pairs kept:", con.execute("SELECT COUNT(*) FROM pair_index_test").fetchone()[0])

con.execute("""
CREATE OR REPLACE TABLE pair_postings_test AS
SELECT p.field, p.token_a, p.token_b, a.entity_id
FROM pair_index_test p
JOIN tokens_top_test a ON a.field = p.field AND a.token = p.token_a
JOIN tokens_top_test b ON b.field = p.field AND b.token = p.token_b AND b.entity_id = a.entity_id
""")
print("pair_postings_test rows:", con.execute("SELECT COUNT(*) FROM pair_postings_test").fetchone()[0])

In [ ]:
con.execute("CREATE OR REPLACE TABLE tokens_test_s1 (entity_id VARCHAR, field VARCHAR, token VARCHAR)")

def load_tokens_chunked_s1(df, field_name, col, table_name, chunk_size=CHUNK_SIZE):
    n_chunks = int(np.ceil(len(df) / chunk_size))
    for i in range(n_chunks):
        chunk = df.iloc[i*chunk_size:(i+1)*chunk_size]
        tmp = pd.DataFrame({
            'entity_id': chunk['entity_id'].astype(str).values,
            'token': chunk[col].apply(tokenize).values
        })
        tmp = tmp.explode('token').dropna(subset=['token'])
        tmp['field'] = field_name
        con.append(table_name, tmp[['entity_id', 'field', 'token']])
        del chunk, tmp
        gc.collect()
        if i % 3 == 0:
            print(f"  {field_name}: chunk {i+1}/{n_chunks}")

t0 = time.time()
load_tokens_chunked_s1(test_s1, 'N', 'business_name', 'tokens_test_s1')
load_tokens_chunked_s1(test_s1, 'A', 'business_address', 'tokens_test_s1')
print(f"tokens_test_s1 load: {(time.time()-t0)/60:.1f} min, rows:",
      con.execute("SELECT COUNT(*) FROM tokens_test_s1").fetchone()[0])

In [ ]:
# Rank S1-test tokens by the TEST-SIDE doc_freq (token_freq_test) -- same rarity source as candidates,
# so it's internally consistent for this test run.
con.execute(f"""
CREATE OR REPLACE TABLE tokens_top_test_s1 AS
SELECT entity_id, field, token, doc_freq FROM (
    SELECT t.entity_id, t.field, t.token,
           COALESCE(f.doc_freq, 999999999) AS doc_freq,
           ROW_NUMBER() OVER (PARTITION BY t.entity_id, t.field
                              ORDER BY COALESCE(f.doc_freq, 999999999) ASC, t.token) AS rnk
    FROM tokens_test_s1 t LEFT JOIN token_freq_test f USING (token)
) WHERE rnk <= {MAX_TOKENS}
""")
print("tokens_top_test_s1 rows:", con.execute("SELECT COUNT(*) FROM tokens_top_test_s1").fetchone()[0])

con.execute("""
CREATE OR REPLACE TABLE pairs_s1_test AS
SELECT a.entity_id, a.field, a.token AS token_a, b.token AS token_b
FROM tokens_top_test_s1 a JOIN tokens_top_test_s1 b
  ON a.entity_id = b.entity_id AND a.field = b.field AND a.token < b.token
""")
print("pairs_s1_test rows:", con.execute("SELECT COUNT(*) FROM pairs_s1_test").fetchone()[0])

In [ ]:
TOP_K = 200
BATCH_SIZE = 20_000

con.execute("DROP TABLE IF EXISTS candidates_final_test")
con.execute("""CREATE TABLE candidates_final_test (
    s1_entity_id VARCHAR, cand_entity_id VARCHAR, best_doc_freq BIGINT
)""")

all_ids_test = con.execute("SELECT DISTINCT entity_id FROM pairs_s1_test").fetchdf()['entity_id'].tolist()
n_batches = int(np.ceil(len(all_ids_test) / BATCH_SIZE))
print(f"{len(all_ids_test):,} entities, {n_batches} batches")

t0 = time.time()
for i in range(n_batches):
    batch_ids = all_ids_test[i*BATCH_SIZE:(i+1)*BATCH_SIZE]
    batch_ids_df = pd.DataFrame({'entity_id': batch_ids})
    con.register('batch_ids_view', batch_ids_df)

    con.execute(f"""
    INSERT INTO candidates_final_test
    WITH joined AS (
        SELECT ps.entity_id AS s1_entity_id, pp.entity_id AS cand_entity_id,
               MIN(pi.doc_freq) AS best_freq
        FROM pairs_s1_test ps
        JOIN batch_ids_view b ON b.entity_id = ps.entity_id
        JOIN pair_index_test pi ON pi.field = ps.field AND pi.token_a = ps.token_a AND pi.token_b = ps.token_b
        JOIN pair_postings_test pp ON pp.field = pi.field AND pp.token_a = pi.token_a AND pp.token_b = pi.token_b
        GROUP BY ps.entity_id, pp.entity_id
    ),
    ranked AS (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY s1_entity_id ORDER BY best_freq ASC, cand_entity_id ASC) AS rnk
        FROM joined
    )
    SELECT s1_entity_id, cand_entity_id, best_freq FROM ranked WHERE rnk <= {TOP_K}
    """)
    con.unregister('batch_ids_view')
    gc.collect()

    if (i+1) % 10 == 0 or i == n_batches-1:
        print(f"  batch {i+1}/{n_batches}, {time.time()-t0:.0f}s elapsed")

print(f"build: {(time.time()-t0)/60:.1f} min")
n = con.execute("SELECT COUNT(*) FROM candidates_final_test").fetchone()[0]
print(f"rows: {n:,}")

In [ ]:
t0 = time.time()
con.execute(f"""
CREATE OR REPLACE TABLE raw_strings_cand_test AS
SELECT CAST(entity_id AS VARCHAR) AS entity_id, business_name, business_address
FROM read_csv_auto('{DATA_ROOT}/dataset/test/test_source2.tsv', delim='\t', header=true)
UNION ALL
SELECT CAST(entity_id AS VARCHAR) AS entity_id, business_name, business_address
FROM read_csv_auto('{DATA_ROOT}/dataset/test/test_source3.tsv', delim='\t', header=true)
""")
print(f"raw_strings_cand_test: {time.time()-t0:.1f}s, rows: {con.execute('SELECT COUNT(*) FROM raw_strings_cand_test').fetchone()[0]:,}")

con.register('test_s1_pd', test_s1)
con.execute("""
CREATE OR REPLACE TABLE raw_strings_test_s1 AS
SELECT CAST(entity_id AS VARCHAR) AS entity_id, business_name, business_address FROM test_s1_pd
""")
print("raw_strings_test_s1 rows:", con.execute("SELECT COUNT(*) FROM raw_strings_test_s1").fetchone()[0])

In [ ]:
t0 = time.time()
con.execute("""
CREATE OR REPLACE TABLE token_counts_test_s1 AS
SELECT entity_id, field, COUNT(DISTINCT token) AS n_tok
FROM tokens_test_s1 GROUP BY entity_id, field
""")
con.execute("""
CREATE OR REPLACE TABLE token_counts_test_cand AS
SELECT entity_id, field, COUNT(DISTINCT token) AS n_tok
FROM tokens_test GROUP BY entity_id, field
""")
print(f"token_counts: {(time.time()-t0)/60:.1f} min")

In [ ]:
JACCARD_TOKEN_FREQ_CAP = 5000
BATCH_SIZE = 50_000

con.execute("DROP TABLE IF EXISTS token_intersect_test")
con.execute("CREATE TABLE token_intersect_test (s1_entity_id VARCHAR, cand_entity_id VARCHAR, field VARCHAR, n_shared BIGINT)")

all_ids_test = con.execute("SELECT DISTINCT entity_id FROM pairs_s1_test").fetchdf()['entity_id'].tolist()
n_batches = int(np.ceil(len(all_ids_test) / BATCH_SIZE))
print(f"{len(all_ids_test):,} entities, {n_batches} batches")

t0 = time.time()
for i in range(n_batches):
    batch_ids = all_ids_test[i*BATCH_SIZE:(i+1)*BATCH_SIZE]
    con.execute("CREATE OR REPLACE TEMP TABLE batch_ids (entity_id VARCHAR)")
    con.executemany("INSERT INTO batch_ids VALUES (?)", [(x,) for x in batch_ids])

    con.execute(f"""
    INSERT INTO token_intersect_test
    SELECT c.s1_entity_id, c.cand_entity_id, s.field, COUNT(DISTINCT s.token) AS n_shared
    FROM candidates_final_test c
    JOIN batch_ids b ON b.entity_id = c.s1_entity_id
    JOIN tokens_test_s1 s ON s.entity_id = c.s1_entity_id
    JOIN token_freq_test f ON f.token = s.token AND f.doc_freq <= {JACCARD_TOKEN_FREQ_CAP}
    JOIN tokens_test t ON t.entity_id = c.cand_entity_id AND t.field = s.field AND t.token = s.token
    GROUP BY c.s1_entity_id, c.cand_entity_id, s.field
    """)
    if (i+1) % 5 == 0 or i == n_batches-1:
        print(f"  batch {i+1}/{n_batches}, {time.time()-t0:.0f}s elapsed")

print(f"total build: {(time.time()-t0)/60:.1f} min")
n = con.execute("SELECT COUNT(*) FROM token_intersect_test").fetchone()[0]
print(f"rows: {n:,}")

In [ ]:
t0 = time.time()
con.execute("""
CREATE OR REPLACE TABLE jaccard_features_test AS
SELECT
    c.s1_entity_id, c.cand_entity_id,
    COALESCE(tn.n_shared, 0) AS name_shared,
    COALESCE(ta.n_shared, 0) AS addr_shared,
    cs.n_tok AS name_n_tok_s1, cc.n_tok AS name_n_tok_cand,
    as1.n_tok AS addr_n_tok_s1, ac.n_tok AS addr_n_tok_cand,
    CASE WHEN (cs.n_tok + cc.n_tok - COALESCE(tn.n_shared,0)) > 0
         THEN COALESCE(tn.n_shared,0)::DOUBLE / (cs.n_tok + cc.n_tok - COALESCE(tn.n_shared,0))
         ELSE 0 END AS name_jaccard,
    CASE WHEN (as1.n_tok + ac.n_tok - COALESCE(ta.n_shared,0)) > 0
         THEN COALESCE(ta.n_shared,0)::DOUBLE / (as1.n_tok + ac.n_tok - COALESCE(ta.n_shared,0))
         ELSE 0 END AS addr_jaccard,
    CASE WHEN COALESCE(tn.n_shared,0) > 0 THEN 1 ELSE 0 END AS name_has_shared_rare_token,
    CASE WHEN COALESCE(ta.n_shared,0) > 0 THEN 1 ELSE 0 END AS addr_has_shared_rare_token
FROM candidates_final_test c
LEFT JOIN token_intersect_test tn ON tn.s1_entity_id = c.s1_entity_id AND tn.cand_entity_id = c.cand_entity_id AND tn.field = 'N'
LEFT JOIN token_intersect_test ta ON ta.s1_entity_id = c.s1_entity_id AND ta.cand_entity_id = c.cand_entity_id AND ta.field = 'A'
LEFT JOIN token_counts_test_s1 cs ON cs.entity_id = c.s1_entity_id AND cs.field = 'N'
LEFT JOIN token_counts_test_cand cc ON cc.entity_id = c.cand_entity_id AND cc.field = 'N'
LEFT JOIN token_counts_test_s1 as1 ON as1.entity_id = c.s1_entity_id AND as1.field = 'A'
LEFT JOIN token_counts_test_cand ac ON ac.entity_id = c.cand_entity_id AND ac.field = 'A'
""")
print(f"jaccard_features_test: {(time.time()-t0)/60:.1f} min, rows: {con.execute('SELECT COUNT(*) FROM jaccard_features_test').fetchone()[0]:,}")

In [ ]:
t0 = time.time()
con.execute("""
CREATE OR REPLACE TABLE exact_match_features_test AS
SELECT s1_entity_id, cand_entity_id,
       CASE WHEN name_jaccard = 1 AND name_n_tok_s1 = name_n_tok_cand THEN 1 ELSE 0 END AS name_exact_approx,
       CASE WHEN addr_jaccard = 1 AND addr_n_tok_s1 = addr_n_tok_cand THEN 1 ELSE 0 END AS addr_exact_approx
FROM jaccard_features_test
""")
n = con.execute("SELECT COUNT(*) FROM exact_match_features_test").fetchone()[0]
print(f"exact_match_features_test: {(time.time()-t0)/60:.1f} min, rows: {n:,}")

In [ ]:
import rapidfuzz
from rapidfuzz.distance import JaroWinkler

BATCH_SIZE = 20_000

con.execute("DROP TABLE IF EXISTS jw_features_test")
con.execute("CREATE TABLE jw_features_test (s1_entity_id VARCHAR, cand_entity_id VARCHAR, name_jw DOUBLE, addr_jw DOUBLE)")

all_ids_test = con.execute("SELECT DISTINCT entity_id FROM pairs_s1_test").fetchdf()['entity_id'].tolist()
n_batches = int(np.ceil(len(all_ids_test) / BATCH_SIZE))
print(f"{len(all_ids_test):,} entities, {n_batches} batches")

t0 = time.time()
for i in range(n_batches):
    batch_ids = all_ids_test[i*BATCH_SIZE:(i+1)*BATCH_SIZE]
    batch_ids_df = pd.DataFrame({'entity_id': batch_ids})
    con.register('batch_ids_view', batch_ids_df)

    pairs_df = con.execute("""
        SELECT c.s1_entity_id, c.cand_entity_id,
               s1n.business_name AS s1_name, s1a.business_address AS s1_addr,
               cn.business_name AS cand_name, ca.business_address AS cand_addr
        FROM candidates_final_test c
        JOIN batch_ids_view b ON b.entity_id = c.s1_entity_id
        JOIN raw_strings_test_s1 s1n ON s1n.entity_id = c.s1_entity_id
        JOIN raw_strings_test_s1 s1a ON s1a.entity_id = c.s1_entity_id
        JOIN raw_strings_cand_test cn ON cn.entity_id = c.cand_entity_id
        JOIN raw_strings_cand_test ca ON ca.entity_id = c.cand_entity_id
    """).fetchdf()

    pairs_df['name_jw'] = [JaroWinkler.normalized_similarity(str(a), str(b)) for a, b in zip(pairs_df['s1_name'], pairs_df['cand_name'])]
    pairs_df['addr_jw'] = [JaroWinkler.normalized_similarity(str(a), str(b)) for a, b in zip(pairs_df['s1_addr'], pairs_df['cand_addr'])]

    con.append('jw_features_test', pairs_df[['s1_entity_id', 'cand_entity_id', 'name_jw', 'addr_jw']])
    con.unregister('batch_ids_view')
    del pairs_df
    gc.collect()

    if (i+1) % 5 == 0 or i == n_batches-1:
        print(f"  batch {i+1}/{n_batches}, {time.time()-t0:.0f}s elapsed")

print(f"build: {(time.time()-t0)/60:.1f} min")
n = con.execute("SELECT COUNT(*) FROM jw_features_test").fetchone()[0]
print(f"rows: {n:,} (expected 332,087,908)")

In [ ]:
# Step 1: TF-IDF weights for test candidates (S2+S3), using tokens_test + token_freq_test
t0 = time.time()
n_cand_docs_test = con.execute("SELECT COUNT(DISTINCT entity_id) FROM tokens_test").fetchone()[0]
print(f"N candidate docs (test S2+S3): {n_cand_docs_test:,}")

con.execute(f"""
CREATE OR REPLACE TABLE tfidf_cand_test AS
SELECT t.entity_id, t.field, t.token,
       COUNT(*) AS tf,
       LN({n_cand_docs_test}.0 / f.doc_freq) AS idf
FROM tokens_test t
JOIN token_freq_test f ON f.token = t.token
GROUP BY t.entity_id, t.field, t.token, f.doc_freq
""")
print(f"tfidf_cand_test: {(time.time()-t0)/60:.1f} min, rows: {con.execute('SELECT COUNT(*) FROM tfidf_cand_test').fetchone()[0]:,}")

In [ ]:
# Step 2: TF-IDF weights for test S1, using its own doc-frequency (test_s1's own vocabulary)
t0 = time.time()
con.execute("""
CREATE OR REPLACE TABLE token_freq_test_s1 AS
SELECT token, COUNT(DISTINCT entity_id) AS doc_freq FROM tokens_test_s1 GROUP BY token
""")
n_s1_docs_test = con.execute("SELECT COUNT(DISTINCT entity_id) FROM tokens_test_s1").fetchone()[0]
print(f"token_freq_test_s1: {(time.time()-t0)/60:.1f} min, N_s1_docs_test={n_s1_docs_test:,}")

con.execute(f"""
CREATE OR REPLACE TABLE tfidf_test_s1 AS
SELECT t.entity_id, t.field, t.token,
       COUNT(*) AS tf,
       LN({n_s1_docs_test}.0 / f.doc_freq) AS idf
FROM tokens_test_s1 t
JOIN token_freq_test_s1 f ON f.token = t.token
GROUP BY t.entity_id, t.field, t.token, f.doc_freq
""")
print(f"tfidf_test_s1: {con.execute('SELECT COUNT(*) FROM tfidf_test_s1').fetchone()[0]:,} rows")

In [ ]:
# Step 3: norms
t0 = time.time()
con.execute("""
CREATE OR REPLACE TABLE tfidf_norms_cand_test AS
SELECT entity_id, field, SQRT(SUM((tf*idf)*(tf*idf))) AS norm
FROM tfidf_cand_test GROUP BY entity_id, field
""")
con.execute("""
CREATE OR REPLACE TABLE tfidf_norms_test_s1 AS
SELECT entity_id, field, SQRT(SUM((tf*idf)*(tf*idf))) AS norm
FROM tfidf_test_s1 GROUP BY entity_id, field
""")
print(f"norms built: {(time.time()-t0)/60:.1f} min")
print("norms_cand_test rows:", con.execute("SELECT COUNT(*) FROM tfidf_norms_cand_test").fetchone()[0])
print("norms_test_s1 rows:", con.execute("SELECT COUNT(*) FROM tfidf_norms_test_s1").fetchone()[0])

In [ ]:
con.execute("DROP TABLE IF EXISTS tfidf_cosine_test")
con.execute("""CREATE OR REPLACE TABLE tfidf_cosine_test (
    s1_entity_id VARCHAR, cand_entity_id VARCHAR,
    name_tfidf_cosine DOUBLE, addr_tfidf_cosine DOUBLE
)""")

all_ids_test = con.execute("SELECT DISTINCT entity_id FROM pairs_s1_test").fetchdf()['entity_id'].tolist()
BATCH_SIZE = 20_000
n_batches = int(np.ceil(len(all_ids_test) / BATCH_SIZE))
print(f"{len(all_ids_test):,} entities, {n_batches} batches")

t0 = time.time()
for i in range(n_batches):
    batch_ids = all_ids_test[i*BATCH_SIZE:(i+1)*BATCH_SIZE]
    batch_ids_df = pd.DataFrame({'entity_id': batch_ids})
    con.register('batch_ids_view', batch_ids_df)

    con.execute("""
    INSERT INTO tfidf_cosine_test
    WITH batch_pairs AS (
        SELECT c.s1_entity_id, c.cand_entity_id
        FROM candidates_final_test c
        JOIN batch_ids_view b ON b.entity_id = c.s1_entity_id
    ),
    batch_tfidf_s1 AS (SELECT t.* FROM tfidf_test_s1 t JOIN batch_ids_view b ON b.entity_id = t.entity_id),
    batch_tfidf_cand AS (
        SELECT t.* FROM tfidf_cand_test t
        JOIN (SELECT DISTINCT cand_entity_id FROM batch_pairs) bc ON bc.cand_entity_id = t.entity_id
    ),
    dot_products AS (
        SELECT bp.s1_entity_id, bp.cand_entity_id, s.field,
               SUM(s.tf * s.idf * c.tf * c.idf) AS dot
        FROM batch_pairs bp
        JOIN batch_tfidf_s1 s ON s.entity_id = bp.s1_entity_id
        JOIN batch_tfidf_cand c ON c.entity_id = bp.cand_entity_id AND c.field = s.field AND c.token = s.token
        GROUP BY bp.s1_entity_id, bp.cand_entity_id, s.field
    )
    SELECT
        bp.s1_entity_id, bp.cand_entity_id,
        COALESCE(dn.dot / NULLIF(ns1n.norm * ncn.norm, 0), 0) AS name_tfidf_cosine,
        COALESCE(da.dot / NULLIF(ns1a.norm * nca.norm, 0), 0) AS addr_tfidf_cosine
    FROM batch_pairs bp
    LEFT JOIN dot_products dn ON dn.s1_entity_id = bp.s1_entity_id AND dn.cand_entity_id = bp.cand_entity_id AND dn.field = 'N'
    LEFT JOIN dot_products da ON da.s1_entity_id = bp.s1_entity_id AND da.cand_entity_id = bp.cand_entity_id AND da.field = 'A'
    LEFT JOIN tfidf_norms_test_s1 ns1n ON ns1n.entity_id = bp.s1_entity_id AND ns1n.field = 'N'
    LEFT JOIN tfidf_norms_cand_test ncn ON ncn.entity_id = bp.cand_entity_id AND ncn.field = 'N'
    LEFT JOIN tfidf_norms_test_s1 ns1a ON ns1a.entity_id = bp.s1_entity_id AND ns1a.field = 'A'
    LEFT JOIN tfidf_norms_cand_test nca ON nca.entity_id = bp.cand_entity_id AND nca.field = 'A'
    """)
    con.unregister('batch_ids_view')
    gc.collect()

    if (i+1) % 10 == 0 or i == n_batches-1:
        print(f"  batch {i+1}/{n_batches}, {time.time()-t0:.0f}s elapsed")

print(f"build: {(time.time()-t0)/60:.1f} min")
n = con.execute("SELECT COUNT(*) FROM tfidf_cosine_test").fetchone()[0]
print(f"rows: {n:,} (expected 332,087,908)")

In [ ]:
# Check current limit first
print(con.execute("SELECT current_setting('memory_limit')").fetchone())

In [ ]:
# Interrupt the stuck cell first (Kernel > Interrupt or the stop button), then run this
con.close()
con = duckdb.connect('/home/ec2-user/SageMaker/blocking.duckdb')
con.execute("PRAGMA memory_limit='20GB'")
print(con.execute("SELECT current_setting('memory_limit')").fetchone())

In [ ]:
import pandas as pd, numpy as np, os, gc, time, shutil
import duckdb
DATA_ROOT = "/home/ec2-user/SageMaker/data"
con = duckdb.connect('/home/ec2-user/SageMaker/blocking.duckdb')
con.execute("PRAGMA memory_limit='8GB'")

import psutil
mem = psutil.virtual_memory()
print(f"available RAM: {mem.available/1e9:.2f} GB / {mem.total/1e9:.2f} GB ({mem.percent}% used)")

In [ ]:
# Confirm all durable tables survived (they should — nothing here was ever in-memory only)
tables = [t[0] for t in con.execute("SHOW TABLES").fetchall()]
print([t for t in tables if 'test' in t])

In [ ]:
# Raise the limit — but to a measured middle value this time, not the max, learning from earlier
con.execute("PRAGMA memory_limit='16GB'")
con.execute("SET preserve_insertion_order=false")  # reduces memory pressure per the error's own suggestion
print(con.execute("SELECT current_setting('memory_limit')").fetchone())

In [ ]:
# Restart kernel first if the current cell is still stuck, then reconnect (memory_limit=8GB as before)
# Export each feature table to parquet - straight scans, no joins, should be fast like everything else tonight
t0 = time.time()
for tbl in ['jaccard_features_test', 'exact_match_features_test', 'jw_features_test', 'tfidf_cosine_test']:
    con.execute(f"COPY {tbl} TO '{DATA_ROOT}/{tbl}.parquet' (FORMAT PARQUET)")
    print(f"  {tbl} exported: {time.time()-t0:.0f}s elapsed")
print(f"all exports: {(time.time()-t0)/60:.1f} min")

In [ ]:
t0 = time.time()
info = con.execute(f"SELECT COUNT(*) FROM read_parquet('{DATA_ROOT}/jaccard_features_test.parquet')").fetchone()
print(f"plain count: {time.time()-t0:.1f}s, rows: {info[0]:,}")

In [ ]:
t0 = time.time()
test_tiny = con.execute(f"""
    SELECT jf.s1_entity_id, jf.cand_entity_id
    FROM read_parquet('{DATA_ROOT}/jaccard_features_test.parquet') jf
    WHERE jf.s1_entity_id IN (
        SELECT entity_id FROM pairs_s1_test LIMIT 1000
    )
""").fetchdf()
elapsed = time.time() - t0
print(f"IN-subquery filter, 1000 entities: {elapsed:.1f}s, rows: {len(test_tiny):,}")

In [ ]:
sample = con.execute("SELECT s1_entity_id FROM jaccard_features_test LIMIT 5").fetchdf()
print(sample)
print(con.execute("SELECT typeof(s1_entity_id) FROM jaccard_features_test LIMIT 1").fetchone())

In [ ]:
t0 = time.time()
for tbl in ['jaccard_features_test', 'exact_match_features_test', 'jw_features_test', 'tfidf_cosine_test']:
    con.execute(f"""
    COPY (
        SELECT
            CAST(regexp_replace(s1_entity_id, '^[A-Za-z0-9]*-', '') AS BIGINT) AS s1_id,
            CAST(regexp_replace(cand_entity_id, '^[A-Za-z0-9]*-', '') AS BIGINT) AS cand_id,
            * EXCLUDE (s1_entity_id, cand_entity_id)
        FROM {tbl}
        ORDER BY s1_id
    ) TO '{DATA_ROOT}/{tbl}_sorted.parquet' (FORMAT PARQUET, ROW_GROUP_SIZE 500000)
    """)
    print(f"  {tbl} recast+sorted: {time.time()-t0:.0f}s elapsed")
print(f"all done: {(time.time()-t0)/60:.1f} min")

In [ ]:
con.execute("PRAGMA memory_limit='10GB'")

t0 = time.time()
con.execute(f"""
CREATE OR REPLACE TABLE step1 AS
SELECT jf.s1_id, jf.cand_id, jf.name_jaccard, jf.name_has_shared_rare_token,
       jf.addr_jaccard, jf.addr_has_shared_rare_token,
       em.name_exact_approx, em.addr_exact_approx
FROM read_parquet('{DATA_ROOT}/jaccard_features_test_sorted.parquet') jf
JOIN read_parquet('{DATA_ROOT}/exact_match_features_test_sorted.parquet') em
  ON em.s1_id = jf.s1_id AND em.cand_id = jf.cand_id
""")
print(f"step1 (jaccard+exact), full 332M rows, 2-table join: {(time.time()-t0)/60:.1f} min")
n1 = con.execute("SELECT COUNT(*) FROM step1").fetchone()[0]
print(f"step1 rows: {n1:,}")

In [ ]:
t0 = time.time()
con.execute(f"""
CREATE OR REPLACE TABLE step2 AS
SELECT s1.s1_id, s1.cand_id, s1.name_jaccard, s1.name_has_shared_rare_token,
       s1.addr_jaccard, s1.addr_has_shared_rare_token,
       s1.name_exact_approx, s1.addr_exact_approx,
       jw.name_jw, jw.addr_jw
FROM step1 s1
JOIN read_parquet('{DATA_ROOT}/jw_features_test_sorted.parquet') jw
  ON jw.s1_id = s1.s1_id AND jw.cand_id = s1.cand_id
""")
print(f"step2 (+jw): {(time.time()-t0)/60:.1f} min")
n2 = con.execute("SELECT COUNT(*) FROM step2").fetchone()[0]
print(f"step2 rows: {n2:,}")

In [ ]:
t0 = time.time()
con.execute(f"""
CREATE OR REPLACE TABLE test_matrix_full AS
SELECT s2.s1_id, s2.cand_id, s2.name_jaccard, s2.name_has_shared_rare_token,
       s2.addr_jaccard, s2.addr_has_shared_rare_token,
       s2.name_exact_approx, s2.addr_exact_approx,
       s2.name_jw, s2.addr_jw,
       tc.name_tfidf_cosine, tc.addr_tfidf_cosine
FROM step2 s2
JOIN read_parquet('{DATA_ROOT}/tfidf_cosine_test_sorted.parquet') tc
  ON tc.s1_id = s2.s1_id AND tc.cand_id = s2.cand_id
""")
print(f"test_matrix_full (+tfidf, final): {(time.time()-t0)/60:.1f} min")
n = con.execute("SELECT COUNT(*) FROM test_matrix_full").fetchone()[0]
print(f"final rows: {n:,} (expected ~332,087,908)")

In [ ]:
dupes = con.execute("""
    SELECT s1_id, cand_id, COUNT(*) as c
    FROM test_matrix_full
    GROUP BY s1_id, cand_id
    HAVING COUNT(*) > 1
    ORDER BY c DESC
    LIMIT 10
""").fetchdf()
print(dupes)
print(f"total dupe groups: {con.execute('SELECT COUNT(*) FROM (SELECT s1_id, cand_id FROM test_matrix_full GROUP BY s1_id, cand_id HAVING COUNT(*) > 1)').fetchone()[0]}")

In [ ]:
con.execute("CREATE OR REPLACE TABLE test_matrix_full AS SELECT DISTINCT * FROM test_matrix_full")
n_final = con.execute("SELECT COUNT(*) FROM test_matrix_full").fetchone()[0]
print(f"final rows after dedup: {n_final:,} (expected 332,087,908)")

In [ ]:
# Find which (s1_id, cand_id) pairs still have >1 row and see if their feature values actually differ
dupes2 = con.execute("""
    SELECT s1_id, cand_id, COUNT(*) as c
    FROM test_matrix_full
    GROUP BY s1_id, cand_id
    HAVING COUNT(*) > 1
    ORDER BY c DESC
    LIMIT 5
""").fetchdf()
print(dupes2)

In [ ]:
# Then look at the actual rows for one of those pairs to see which column differs
if len(dupes2) > 0:
    s1, cand = dupes2.iloc[0]['s1_id'], dupes2.iloc[0]['cand_id']
    print(con.execute(f"SELECT * FROM test_matrix_full WHERE s1_id={s1} AND cand_id={cand}").fetchdf())

In [ ]:
for tbl, id_cols in [
    ('jaccard_features_test', 's1_entity_id, cand_entity_id'),
    ('jw_features_test', 's1_entity_id, cand_entity_id'),
    ('tfidf_cosine_test', 's1_entity_id, cand_entity_id'),
]:
    r = con.execute(f"""
        SELECT * FROM {tbl}
        WHERE regexp_replace(s1_entity_id, '^[A-Za-z0-9]*-', '') = '983530310'
          AND regexp_replace(cand_entity_id, '^[A-Za-z0-9]*-', '') = '335309159'
    """).fetchdf()
    print(f"--- {tbl} ---")
    print(r)

In [ ]:
# Check exactly how many collisions exist across the full dataset before deciding next steps
collisions = con.execute(f"""
    SELECT regexp_replace(cand_entity_id, '^[A-Za-z0-9]*-', '') AS num_id, COUNT(DISTINCT cand_entity_id) AS distinct_prefixed
    FROM jaccard_features_test
    GROUP BY num_id
    HAVING COUNT(DISTINCT cand_entity_id) > 1
""").fetchdf()
print(f"colliding numeric ids: {len(collisions)}")

In [ ]:
t0 = time.time()
con.execute("""
CREATE OR REPLACE TABLE step1 AS
SELECT jf.s1_entity_id, jf.cand_entity_id, jf.name_jaccard, jf.name_has_shared_rare_token,
       jf.addr_jaccard, jf.addr_has_shared_rare_token,
       em.name_exact_approx, em.addr_exact_approx
FROM jaccard_features_test jf
JOIN exact_match_features_test em
  ON em.s1_entity_id = jf.s1_entity_id AND em.cand_entity_id = jf.cand_entity_id
""")
print(f"step1: {(time.time()-t0)/60:.1f} min, rows: {con.execute('SELECT COUNT(*) FROM step1').fetchone()[0]:,}")

In [ ]:
t0 = time.time()
con.execute("""
CREATE OR REPLACE TABLE step2 AS
SELECT s1.s1_entity_id, s1.cand_entity_id, s1.name_jaccard, s1.name_has_shared_rare_token,
       s1.addr_jaccard, s1.addr_has_shared_rare_token,
       s1.name_exact_approx, s1.addr_exact_approx,
       jw.name_jw, jw.addr_jw
FROM step1 s1
JOIN jw_features_test jw
  ON jw.s1_entity_id = s1.s1_entity_id AND jw.cand_entity_id = s1.cand_entity_id
""")
print(f"step2: {(time.time()-t0)/60:.1f} min, rows: {con.execute('SELECT COUNT(*) FROM step2').fetchone()[0]:,}")

In [ ]:
t0 = time.time()
con.execute("""
CREATE OR REPLACE TABLE test_matrix_full AS
SELECT s2.s1_entity_id, s2.cand_entity_id, s2.name_jaccard, s2.name_has_shared_rare_token,
       s2.addr_jaccard, s2.addr_has_shared_rare_token,
       s2.name_exact_approx, s2.addr_exact_approx,
       s2.name_jw, s2.addr_jw,
       tc.name_tfidf_cosine, tc.addr_tfidf_cosine
FROM step2 s2
JOIN tfidf_cosine_test tc
  ON tc.s1_entity_id = s2.s1_entity_id AND tc.cand_entity_id = s2.cand_entity_id
""")
n = con.execute("SELECT COUNT(*) FROM test_matrix_full").fetchone()[0]
print(f"final: {(time.time()-t0)/60:.1f} min, rows: {n:,} (expected 332,087,908)")

In [ ]:
con.execute("PRAGMA memory_limit='8GB'")

all_s1 = con.execute("SELECT DISTINCT entity_id FROM pairs_s1_test").fetchdf()['entity_id'].tolist()
BATCH_SIZE = 100_000
n_batches = int(np.ceil(len(all_s1) / BATCH_SIZE))
print(f"{len(all_s1):,} entities, {n_batches} batches")

first = True
t0 = time.time()
for i in range(n_batches):
    batch = all_s1[i*BATCH_SIZE:(i+1)*BATCH_SIZE]
    batch_df = pd.DataFrame({'entity_id': batch})
    con.register('batch_view', batch_df)
    chunk = con.execute("""
        SELECT b.entity_id AS source1_entity_id,
               COALESCE(string_agg(c.cand_entity_id, ','), '') AS candidate_entity_ids
        FROM batch_view b
        LEFT JOIN candidates_final_test c ON c.s1_entity_id = b.entity_id
        GROUP BY b.entity_id
    """).fetchdf()
    con.unregister('batch_view')
    chunk.to_csv(f"{DATA_ROOT}/candidate_pairs.tsv", sep='\t', index=False, mode='w' if first else 'a', header=first)
    first = False
    del chunk
    gc.collect()
    if (i+1) % 5 == 0 or i == n_batches-1:
        print(f"  batch {i+1}/{n_batches}, {time.time()-t0:.0f}s elapsed")
print(f"done: {(time.time()-t0)/60:.1f} min")
print(open(f"{DATA_ROOT}/candidate_pairs.tsv").readline())